# Six Looks, One Harvest — final kharif 2025 yield forecast for Sokhda, Vadodara

**ANRF AISEHack 2.0 · Round 3 · Team 8bit** (Harsh Thummar, Viraj Suhagiya)

966 farm plots · 447.5 ha · Capella X-band HH stripmap SLC, six passes:
**6 Jun / 19 Jun / 14 Aug / 13 Oct / 29 Oct / 12 Nov 2025**

This notebook runs the complete chain from the raw complex SLC to the plot-level
and village-level yield forecasts. Everything geometric and radiometric is
implemented from the Capella extended metadata and the RPC model — no SNAP, no
ISCE, no `gdalwarp`.

**Pipeline**

1. Scene inventory, calibration constants, per-range-sample incidence angle
2. RPC00B forward model, validated against the 225 GCPs in each product
3. Solve the reference height from the imagery (it is *not* the RPC default)
4. Geocode all six passes to a common 2 m UTM-43N γ⁰ grid
5. Bounded inter-pass co-registration and radiometric cross-checks
6. Per-plot zonal statistics for 966 plots × 6 dates
7. Agronomic knowledge base for Vadodara kharif (external data, all sourced)
8. Crop labels: carry-forward, plus an independent 6-pass re-derivation
9. Independent validation against same-day Sentinel-2 NDVI
10. Yield forecast, Monte-Carlo uncertainty, village aggregation

**Key design decision.** X-band HH saturates early against LAI, and on this
village the median Δγ⁰ at peak season is *negative* relative to bare soil —
a closed canopy attenuates a rough tilled surface more than it scatters at HH.
Inverting X-band for absolute biomass would therefore be indefensible. The model
instead uses phenological **timing**, late-season **retention**, and within-field
**evenness**, and anchors the absolute level to district statistics.

## 0 · Environment and paths

In [1]:
import os, sys, json, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

# Works on Kaggle (competition dataset attached) or locally with a DATA/ folder.
CANDIDATES = [
    "/kaggle/input/anrf-aise-hack-2-0-round-3-sar-crop-yield-forecasting",
    "/kaggle/input",
]
# also walk upward from the working directory, so the notebook runs unchanged
# from a subfolder of a local checkout
_p = os.path.abspath(".")
for _ in range(5):
    CANDIDATES += [os.path.join(_p, "DATA"), _p]
    _p = os.path.dirname(_p)
DATA_ROOT = None
for c in CANDIDATES:
    if os.path.isdir(c) and glob.glob(os.path.join(c, "**", "CAPELLA_*"), recursive=True):
        hits = glob.glob(os.path.join(c, "**", "CAPELLA_*"), recursive=True)
        DATA_ROOT = os.path.dirname(sorted(hits)[0])
        break
assert DATA_ROOT, "could not locate the Capella scenes"
FARM_SHP = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*Farms.shp"), recursive=True))[0]
VILL_SHP = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*Village.shp"), recursive=True))[0]
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
GEO = os.path.join(WORKDIR, "geo"); OUT = os.path.join(WORKDIR, "out")
FIG = os.path.join(WORKDIR, "figures")
for d in (GEO, OUT, FIG): os.makedirs(d, exist_ok=True)
print("DATA_ROOT :", DATA_ROOT)
print("farms     :", FARM_SHP)
print("scenes    :", len([p for p in glob.glob(os.path.join(DATA_ROOT, "CAPELLA_*")) if os.path.isdir(p)]))

DATA_ROOT : C:\Users\HARSH\Desktop\aishe round 3 _ 2nd try\DATA
farms     : C:\Users\HARSH\Desktop\aishe round 3 _ 2nd try\DATA\Farm_boundaries_shp\Farm_boundaries_shp\Sokhda_Farms.shp
scenes    : 6


## 1 · SAR core — calibration, geometry, geocoding

Capella delivers SLC as `beta_nought` with `calibration: full`, so

$$\beta^0 = (|DN|\cdot s)^2, \qquad \sigma^0 = \beta^0\sin\theta, \qquad
\gamma^0 = \beta^0\tan\theta$$

γ⁰ is used throughout because the six passes span **28.69°–35.24°** incidence and
γ⁰ is, to first order, the incidence-invariant quantity for volume scatterers.
The local incidence angle per range sample comes from a spherical-Earth solution
on the product state vectors, pinned to the metadata centre value.

In [2]:
"""
sarlib.py -- Capella X-band SLC -> calibrated, geocoded gamma-nought.

Core utilities shared by the Round-3 pipeline:
  * RPC00B rational-polynomial forward model (ground -> image), vectorised.
  * Capella radiometric calibration  (DN -> beta0 -> sigma0 -> gamma0).
  * Per-range-sample local incidence angle from a spherical-Earth solution
    using the product state vectors.
  * Inverse geocoding of the slant-plane image onto a UTM grid by
    super-sampled nearest-neighbour gather + block averaging (this performs
    the spatial multi-look in the map domain).

Radiometry reference: Capella SLC products are delivered with
`collect.image.radiometry == "beta_nought"` and `calibration == "full"`,
so that
        beta0 = (|DN| * scale_factor)^2
        sigma0 = beta0 * sin(theta_inc)
        gamma0 = sigma0 / cos(theta_inc) = beta0 * tan(theta_inc)
gamma0 is used throughout because it is the quantity that is (to first
order) invariant to incidence angle for volume scatterers -- essential
here, because the six passes span theta = 28.7 deg .. 35.2 deg.
"""

from __future__ import annotations

import glob
import json
import os

import numpy as np
import rasterio
from pyproj import Transformer

# --------------------------------------------------------------------------
# constants
# --------------------------------------------------------------------------
UTM43N = "EPSG:32643"
WGS84 = "EPSG:4326"


# --------------------------------------------------------------------------
# RPC00B forward model
# --------------------------------------------------------------------------
def _rpc_terms(L: np.ndarray, P: np.ndarray, H: np.ndarray) -> np.ndarray:
    """20 monomials of the RPC00B cubic, in GDAL/NITF order.

    L = normalised longitude, P = normalised latitude, H = normalised height.
    Returns an array of shape (20, n).
    """
    return np.stack([
        np.ones_like(L),
        L, P, H,
        L * P, L * H, P * H,
        L * L, P * P, H * H,
        P * L * H,
        L ** 3, L * P * P, L * H * H,
        L * L * P, P ** 3, P * H * H,
        L * L * H, P * P * H, H ** 3,
    ])


class RPC:
    """Ground (lon, lat, h) -> image (line, sample) rational polynomial."""

    def __init__(self, gdal_rpc: dict):
        g = {k: v for k, v in gdal_rpc.items()}

        def _f(key):
            return float(g[key])

        def _c(key):
            v = g[key]
            if isinstance(v, str):
                v = [float(x) for x in v.split()]
            return np.asarray(v, dtype=np.float64)

        self.line_off, self.line_scale = _f("LINE_OFF"), _f("LINE_SCALE")
        self.samp_off, self.samp_scale = _f("SAMP_OFF"), _f("SAMP_SCALE")
        self.lat_off, self.lat_scale = _f("LAT_OFF"), _f("LAT_SCALE")
        self.long_off, self.long_scale = _f("LONG_OFF"), _f("LONG_SCALE")
        self.height_off, self.height_scale = _f("HEIGHT_OFF"), _f("HEIGHT_SCALE")
        self.line_num = _c("LINE_NUM_COEFF")
        self.line_den = _c("LINE_DEN_COEFF")
        self.samp_num = _c("SAMP_NUM_COEFF")
        self.samp_den = _c("SAMP_DEN_COEFF")

    def forward(self, lon, lat, h=None):
        """(lon, lat, height) -> (line, sample). Arrays broadcast; float64."""
        lon = np.asarray(lon, dtype=np.float64)
        lat = np.asarray(lat, dtype=np.float64)
        if h is None:
            h = np.full(lon.shape, self.height_off, dtype=np.float64)
        h = np.broadcast_to(np.asarray(h, dtype=np.float64), lon.shape)

        L = (lon - self.long_off) / self.long_scale
        P = (lat - self.lat_off) / self.lat_scale
        H = (h - self.height_off) / self.height_scale

        T = _rpc_terms(L.ravel(), P.ravel(), H.ravel())      # (20, n)
        line = self.line_off + self.line_scale * ((self.line_num @ T) / (self.line_den @ T))
        samp = self.samp_off + self.samp_scale * ((self.samp_num @ T) / (self.samp_den @ T))
        return line.reshape(lon.shape), samp.reshape(lon.shape)


# --------------------------------------------------------------------------
# scene discovery / metadata
# --------------------------------------------------------------------------
class Scene:
    """One Capella acquisition: paths, STAC properties, product metadata."""

    def __init__(self, folder: str):
        self.folder = folder
        self.name = os.path.basename(folder.rstrip("\\/"))
        # the June-19 directory also ships a stray copy of the June-06 SLC,
        # so the SLC is selected by exact name match with its own folder.
        want = self.name + ".tif"
        cand = [p for p in glob.glob(os.path.join(folder, "*.tif"))
                if os.path.basename(p) == want]
        if not cand:
            raise FileNotFoundError(f"no SLC matching {want}")
        self.slc = cand[0]
        prev = glob.glob(os.path.join(folder, "*_preview.tif"))
        self.preview = prev[0] if prev else None
        stac = [p for p in glob.glob(os.path.join(folder, "*.json"))
                if "extended" not in p and "digest" not in p]
        self.stac = json.load(open(stac[0]))
        self.props = self.stac["properties"]
        self.meta = json.load(open(glob.glob(os.path.join(folder, "*_extended.json"))[0]))
        self.collect = self.meta["collect"]
        self.image = self.collect["image"]

        self.datetime = self.props["datetime"]
        self.date = self.datetime[:10]
        self.scale_factor = float(self.image["scale_factor"])
        self.inc_centre = float(self.image["center_pixel"]["incidence_angle"])
        self.look = self.collect["radar"]["pointing"]
        self.orbit = self.props["sat:orbit_state"]
        self.az = float(self.props["view:azimuth"])
        self.nesz_peak = float(self.image["nesz_peak"])
        self.rows = int(self.image["rows"])
        self.cols = int(self.image["columns"])
        ig = self.image["image_geometry"]
        self.r0 = float(ig["range_to_first_sample"])
        self.dr = float(ig["delta_range_sample"])

    def __repr__(self):
        return (f"<Scene {self.date} inc={self.inc_centre:.1f} "
                f"look={self.look} az={self.az:.0f}>")

    # ---- geometry -------------------------------------------------------
    def incidence_per_column(self) -> np.ndarray:
        """Local incidence angle (radians) for every range sample.

        Spherical-Earth triangle closed on the satellite radius Rs, the local
        Earth radius Re at the scene reference target, and the slant range r.
        The incidence angle is measured from the local *up* direction, i.e.
        the supplement of the apex angle at the target:
            cos(theta_inc) = (Rs^2 - Re^2 - r^2) / (2 * Re * r)
        The spherical approximation reproduces the metadata centre incidence
        to ~0.1 deg; the profile is therefore shifted by a constant so that
        the centre sample matches `center_pixel.incidence_angle` exactly.
        The residual across-range *gradient* (~0.4 deg over the AOI) is what
        this term actually contributes.
        """
        sat = np.asarray(self.image["reference_antenna_position"], dtype=float)
        tgt = np.asarray(self.image["reference_target_position"], dtype=float)
        Rs = np.linalg.norm(sat)
        Re = np.linalg.norm(tgt)
        r = self.r0 + np.arange(self.cols, dtype=np.float64) * self.dr
        cos_i = (Rs ** 2 - Re ** 2 - r ** 2) / (2.0 * Re * r)
        inc = np.arccos(np.clip(cos_i, -1.0, 1.0))
        inc += np.radians(self.inc_centre) - inc[self.cols // 2]
        return inc

    def nesz_per_column(self) -> np.ndarray:
        """Noise-equivalent sigma-zero (linear power) per range sample."""
        c = np.asarray(self.image["nesz_polynomial"]["coefficients"], dtype=float)
        r = self.r0 + np.arange(self.cols, dtype=np.float64) * self.dr
        db = np.polyval(c[::-1], r)
        return 10.0 ** (db / 10.0)

    def rpc(self) -> RPC:
        with rasterio.open(self.slc) as ds:
            return RPC(ds.rpcs.to_gdal())


def find_scenes(data_root: str):
    folders = sorted(glob.glob(os.path.join(data_root, "CAPELLA_*")))
    folders = [f for f in folders if os.path.isdir(f)]
    return [Scene(f) for f in folders]


# --------------------------------------------------------------------------
# inverse geocoding
# --------------------------------------------------------------------------
def make_grid(bounds_utm, res):
    """Snap bounds to a `res`-metre grid; return (transform, width, height)."""
    from rasterio.transform import from_origin
    xmin, ymin, xmax, ymax = bounds_utm
    xmin = np.floor(xmin / res) * res
    ymin = np.floor(ymin / res) * res
    xmax = np.ceil(xmax / res) * res
    ymax = np.ceil(ymax / res) * res
    w = int(round((xmax - xmin) / res))
    h = int(round((ymax - ymin) / res))
    return from_origin(xmin, ymax, res, res), w, h


def geocode_scene(scene: Scene, transform, width, height, res,
                  height_m=None, dem_sample=None, geoid_offset=0.0,
                  ss=4, chunk_rows=64, verbose=True):
    """Geocode one SLC to gamma0 on a UTM grid.

    Each output cell of size `res` is filled with the mean of ss*ss
    super-sampled nearest-neighbour lookups into the slant-plane image; with
    ss=4 at res=2 m that is 16 sub-samples of ~0.9 m native pixels, i.e. an
    effective ~14-look average in the map domain.

    Terrain handling. Pass `dem_sample`, a callable (lon, lat) -> ORTHOMETRIC
    height, together with `geoid_offset` N, and every sub-sample is projected at
    its own ellipsoidal height H_ortho + N. This is terrain-referenced
    geocoding. Falling back to the scalar `height_m` assumes a flat surface,
    which over this AOI (26.6 m of relief) displaces ground range by up to 46 m
    at 30 deg incidence -- comparable to a whole field.

    Returns (gamma0_linear, valid_fraction, nesz_linear) as float32 arrays.
    """
    rpc = scene.rpc()
    if height_m is None:
        height_m = rpc.height_off

    def heights_for(lon, lat):
        if dem_sample is None:
            return height_m
        h = dem_sample(lon, lat) + geoid_offset
        return np.where(np.isfinite(h), h, height_m)

    inc = scene.incidence_per_column()          # (cols,)
    nesz_col = scene.nesz_per_column()          # (cols,)
    tan_inc = np.tan(inc).astype(np.float32)

    to_ll = Transformer.from_crs(UTM43N, WGS84, always_xy=True)

    # ---- source window covering the output grid --------------------------
    xs = transform.c + np.linspace(0, width, 40) * res
    ys = transform.f - np.linspace(0, height, 40) * res
    gx, gy = np.meshgrid(xs, ys)
    glon, glat = to_ll.transform(gx, gy)
    ln, sm = rpc.forward(glon, glat, heights_for(glon, glat))
    pad = 96
    r_lo = max(0, int(np.floor(np.nanmin(ln))) - pad)
    r_hi = min(scene.rows, int(np.ceil(np.nanmax(ln))) + pad)
    c_lo = max(0, int(np.floor(np.nanmin(sm))) - pad)
    c_hi = min(scene.cols, int(np.ceil(np.nanmax(sm))) + pad)
    if verbose:
        print(f"    src window rows {r_lo}:{r_hi} cols {c_lo}:{c_hi} "
              f"({(r_hi-r_lo)*(c_hi-c_lo)/1e6:.1f} Mpx)")

    # ---- read + calibrate the window ------------------------------------
    from rasterio.windows import Window
    with rasterio.open(scene.slc) as ds:
        z = ds.read(1, window=Window(c_lo, r_lo, c_hi - c_lo, r_hi - r_lo))
    sf = scene.scale_factor
    beta0 = (z.real.astype(np.float32) ** 2 + z.imag.astype(np.float32) ** 2) * np.float32(sf * sf)
    del z
    gam = beta0 * tan_inc[c_lo:c_hi][None, :]     # gamma0 = beta0 * tan(theta)
    del beta0
    src_h, src_w = gam.shape

    # noise floor, expressed as gamma0, for the same columns
    nesz_gam = (nesz_col[c_lo:c_hi] / np.sin(inc[c_lo:c_hi]) * np.tan(inc[c_lo:c_hi])).astype(np.float32)

    out = np.zeros((height, width), dtype=np.float32)
    cnt = np.zeros((height, width), dtype=np.float32)
    nz = np.zeros((height, width), dtype=np.float32)

    sub = (np.arange(ss) + 0.5) / ss            # sub-cell centres
    for r0 in range(0, height, chunk_rows):
        r1 = min(height, r0 + chunk_rows)
        # sub-sample coordinate mesh for this row block
        yy = transform.f - (np.repeat(np.arange(r0, r1), ss) + np.tile(sub, r1 - r0)) * res
        xx = transform.c + (np.repeat(np.arange(width), ss) + np.tile(sub, width)) * res
        X, Y = np.meshgrid(xx, yy)
        lon, lat = to_ll.transform(X, Y)
        ln, sm = rpc.forward(lon, lat, heights_for(lon, lat))
        li = np.rint(ln).astype(np.int64) - r_lo
        si = np.rint(sm).astype(np.int64) - c_lo
        ok = (li >= 0) & (li < src_h) & (si >= 0) & (si < src_w)
        li = np.where(ok, li, 0)
        si = np.where(ok, si, 0)
        vals = np.where(ok, gam[li, si], 0.0).astype(np.float32)
        nvals = np.where(ok, nesz_gam[si], 0.0).astype(np.float32)
        okf = ok.astype(np.float32)
        # block-average ss x ss -> one output cell
        nr = r1 - r0
        out[r0:r1] = vals.reshape(nr, ss, width, ss).sum(axis=(1, 3))
        cnt[r0:r1] = okf.reshape(nr, ss, width, ss).sum(axis=(1, 3))
        nz[r0:r1] = nvals.reshape(nr, ss, width, ss).sum(axis=(1, 3))

    with np.errstate(invalid="ignore", divide="ignore"):
        g = np.where(cnt > 0, out / np.maximum(cnt, 1), np.nan).astype(np.float32)
        n = np.where(cnt > 0, nz / np.maximum(cnt, 1), np.nan).astype(np.float32)
    frac = (cnt / (ss * ss)).astype(np.float32)
    return g, frac, n


def geocode_slant_array(scene: Scene, arr, r_lo, c_lo, transform, width, height, res,
                        height_m=None, dem_sample=None, geoid_offset=0.0,
                        ss=2, chunk_rows=64):
    """Geocode an arbitrary real-valued slant-range array onto the UTM grid.

    `arr` is a slant-plane raster whose (0, 0) element corresponds to image
    (row, col) = (r_lo, c_lo) -- i.e. the same window convention geocode_scene
    uses internally. Used for products derived from the SLC (sub-look coherence)
    rather than the backscatter itself.
    """
    rpc = scene.rpc()
    if height_m is None:
        height_m = rpc.height_off

    def heights_for(lon, lat):
        if dem_sample is None:
            return height_m
        h = dem_sample(lon, lat) + geoid_offset
        return np.where(np.isfinite(h), h, height_m)

    to_ll = Transformer.from_crs(UTM43N, WGS84, always_xy=True)
    src_h, src_w = arr.shape
    out = np.zeros((height, width), dtype=np.float32)
    cnt = np.zeros((height, width), dtype=np.float32)
    sub = (np.arange(ss) + 0.5) / ss

    for r0 in range(0, height, chunk_rows):
        r1 = min(height, r0 + chunk_rows)
        yy = transform.f - (np.repeat(np.arange(r0, r1), ss) + np.tile(sub, r1 - r0)) * res
        xx = transform.c + (np.repeat(np.arange(width), ss) + np.tile(sub, width)) * res
        X, Y = np.meshgrid(xx, yy)
        lon, lat = to_ll.transform(X, Y)
        ln, sm = rpc.forward(lon, lat, heights_for(lon, lat))
        li = np.rint(ln).astype(np.int64) - r_lo
        si = np.rint(sm).astype(np.int64) - c_lo
        ok = (li >= 0) & (li < src_h) & (si >= 0) & (si < src_w)
        li = np.where(ok, li, 0); si = np.where(ok, si, 0)
        vals = np.where(ok, arr[li, si], 0.0).astype(np.float32)
        okf = ok.astype(np.float32)
        nr = r1 - r0
        out[r0:r1] = vals.reshape(nr, ss, width, ss).sum(axis=(1, 3))
        cnt[r0:r1] = okf.reshape(nr, ss, width, ss).sum(axis=(1, 3))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnt > 0, out / np.maximum(cnt, 1), np.nan).astype(np.float32)


def to_db(x):
    x = np.asarray(x, dtype=np.float64)
    return 10.0 * np.log10(np.where(x > 0, x, np.nan))


def write_tif(path, arr, transform, crs=UTM43N, nodata=np.nan, dtype="float32"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    h, w = arr.shape
    with rasterio.open(path, "w", driver="GTiff", height=h, width=w, count=1,
                       dtype=dtype, crs=crs, transform=transform, nodata=nodata,
                       compress="deflate", predictor=3, tiled=True,
                       blockxsize=256, blockysize=256) as ds:
        ds.write(arr.astype(dtype), 1)

## 2 · Scene inventory and geometry validation

The RPC implementation is checked against the 225 ground control points that
Capella ships inside each product. Agreement is exact to the printed precision,
which means any later mis-registration is a *height* problem, not a model
problem — and that is exactly what we find next.

In [3]:
scenes = find_scenes(DATA_ROOT)
import rasterio
rows = []
for s in scenes:
    with rasterio.open(s.slc) as ds:
        rpc = RPC(ds.rpcs.to_gdal()); gcps, _ = ds.get_gcps()
    lon = np.array([g.x for g in gcps]); lat = np.array([g.y for g in gcps])
    z = np.array([g.z for g in gcps]); r = np.array([g.row for g in gcps])
    c = np.array([g.col for g in gcps])
    ln, sm = rpc.forward(lon, lat, z)
    inc = np.degrees(s.incidence_per_column())
    rows.append({"date": s.date, "incidence_deg": round(s.inc_centre, 2), "look": s.look,
                 "orbit": s.orbit, "azimuth": round(s.az, 1),
                 "rows": s.rows, "cols": s.cols,
                 "nesz_peak_dB": round(s.nesz_peak, 2),
                 "inc_near_far": f"{inc.min():.2f}-{inc.max():.2f}",
                 "rpc_rms_line_px": round(float(np.sqrt(np.mean((ln - r) ** 2))), 5),
                 "rpc_rms_samp_px": round(float(np.sqrt(np.mean((sm - c) ** 2))), 5)})
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))

      date  incidence_deg  look     orbit  azimuth  rows  cols  nesz_peak_dB inc_near_far  rpc_rms_line_px  rpc_rms_samp_px
2025-06-06          35.24  left ascending    134.7 27192  4682        -26.13  35.05-35.44          0.00000          0.00000
2025-06-19          28.77  left ascending    135.1 27187  3910        -27.76  28.55-28.99          0.00003          0.00002
2025-08-14          28.69  left ascending    135.1 27219  3897        -27.97  28.47-28.91          0.00000          0.00000
2025-10-13          31.53  left ascending    135.0 27241  4244        -27.35  31.31-31.74          0.00000          0.00000
2025-10-29          29.84 right ascending    318.4 27285  4050        -27.74  29.62-30.06          0.00001          0.00000
2025-11-12          29.75  left ascending    135.2 27273  4031        -27.72  29.52-29.97          0.00001          0.00000


## 3 · Terrain referencing — DEM and the datum solution

Capella's RPCs expect **ellipsoidal** heights; the Copernicus GLO-30 DEM stores
**orthometric** (EGM2008) heights. The 225 GCPs embedded in each product carry
ellipsoidal z, so the datum shift `N = z_GCP − H_DEM` is solved rather than assumed.

The within-scene scatter of that difference is the check on the method: had the
GCP z values been a synthetic multi-layer height grid rather than terrain
samples, the scatter would be tens of metres and the solve would be meaningless.

Terrain cannot be neglected here — the farm block carries **26.6 m of relief**,
which a flat-earth height converts into up to **46 m of ground-range
displacement** at 30° incidence, comparable to the width of a whole field.

In [4]:
"""Fetch the Copernicus GLO-30 DEM and solve the ellipsoid/geoid datum shift.

Capella RPCs expect ELLIPSOIDAL heights; Copernicus DEM stores ORTHOMETRIC
(EGM2008) heights. The 225 GCPs embedded in each SLC carry ellipsoidal z, so the
offset N = h_ellipsoidal - H_orthometric can be solved rather than assumed.

This validates the assumption before it is used: if the GCP z values were a
synthetic multi-layer height grid rather than terrain, the within-scene scatter
of (z - H_ortho) would be tens of metres and the solve would be meaningless.
"""
import json, os, sys, urllib.request
import numpy as np
import rasterio
from rasterio.transform import Affine

WORK = WORKDIR
DEM_LOCAL = os.path.join(WORK, "copdem_N22E073.tif")
DEM_URL = ("https://copernicus-dem-30m.s3.amazonaws.com/"
           "Copernicus_DSM_COG_10_N22_00_E073_00_DEM/"
           "Copernicus_DSM_COG_10_N22_00_E073_00_DEM.tif")


def fetch_dem():
    if os.path.exists(DEM_LOCAL) and os.path.getsize(DEM_LOCAL) > 1_000_000:
        print(f"DEM cached: {DEM_LOCAL} ({os.path.getsize(DEM_LOCAL)/1e6:.1f} MB)")
        return DEM_LOCAL
    print("downloading Copernicus GLO-30 tile N22E073 ...")
    urllib.request.urlretrieve(DEM_URL, DEM_LOCAL)
    print(f"  got {os.path.getsize(DEM_LOCAL)/1e6:.1f} MB")
    return DEM_LOCAL


def make_dem_sampler(path):
    ds = rasterio.open(path)
    dem = ds.read(1).astype(np.float64)
    if ds.nodata is not None:
        dem[dem == ds.nodata] = np.nan
    inv = ~ds.transform
    H, W = dem.shape

    def sample(lon, lat):
        """Bilinear sample of orthometric height at geographic coordinates."""
        lon = np.asarray(lon, dtype=np.float64)
        lat = np.asarray(lat, dtype=np.float64)
        c, r = inv * (lon, lat)
        c = np.clip(c - 0.5, 0, W - 1.001)
        r = np.clip(r - 0.5, 0, H - 1.001)
        c0, r0 = np.floor(c).astype(int), np.floor(r).astype(int)
        fc, fr = c - c0, r - r0
        v = (dem[r0, c0] * (1 - fc) * (1 - fr) + dem[r0, c0 + 1] * fc * (1 - fr) +
             dem[r0 + 1, c0] * (1 - fc) * fr + dem[r0 + 1, c0 + 1] * fc * fr)
        return v

    return sample, ds




scenes = find_scenes(DATA_ROOT)
DEM_SAMPLE, _dem_ds = make_dem_sampler(fetch_dem())
offs = []
for s in scenes:
    with rasterio.open(s.slc) as d:
        gcps, _ = d.get_gcps(); rpc = RPC(d.rpcs.to_gdal())
    lon = np.array([g.x for g in gcps]); lat = np.array([g.y for g in gcps])
    z = np.array([g.z for g in gcps])
    row = np.array([g.row for g in gcps]); col = np.array([g.col for g in gcps])
    ho = DEM_SAMPLE(lon, lat); ok = np.isfinite(ho)
    diff = z[ok] - ho[ok]; offs.append(diff.mean())
    ln1, sm1 = rpc.forward(lon, lat, ho + diff.mean())
    ln2, sm2 = rpc.forward(lon, lat, np.full_like(lon, -20.0))
    e1 = float(np.sqrt(np.mean((ln1-row)**2 + (sm1-col)**2)))
    e2 = float(np.sqrt(np.mean((ln2-row)**2 + (sm2-col)**2)))
    print(f"{s.date}: N = {diff.mean():8.3f} m  within-scene sd = {diff.std():5.2f} m   "
          f"RPC round-trip RMSE: DEM {e1:6.3f} px vs constant-height {e2:7.3f} px")
GEOID_N = float(np.mean(offs))
print(f"\nadopted geoid offset N = {GEOID_N:.3f} m "
      f"(scene-to-scene sd {np.std(offs):.3f} m)")

DEM cached: .\copdem_N22E073.tif (46.1 MB)


2025-06-06: N =  -62.075 m  within-scene sd =  3.34 m   RPC round-trip RMSE: DEM  4.421 px vs constant-height  15.131 px
2025-06-19: N =  -62.178 m  within-scene sd =  3.31 m   RPC round-trip RMSE: DEM  4.698 px vs constant-height  15.870 px
2025-08-14: N =  -62.036 m  within-scene sd =  3.19 m   RPC round-trip RMSE: DEM  4.529 px vs constant-height  16.142 px
2025-10-13: N =  -62.044 m  within-scene sd =  3.08 m   RPC round-trip RMSE: DEM  4.257 px vs constant-height  15.660 px
2025-10-29: N =  -61.768 m  within-scene sd =  2.92 m   RPC round-trip RMSE: DEM  4.099 px vs constant-height  16.329 px
2025-11-12: N =  -62.035 m  within-scene sd =  3.25 m   RPC round-trip RMSE: DEM  4.570 px vs constant-height  16.605 px

adopted geoid offset N = -62.023 m (scene-to-scene sd 0.124 m)


## 3b · Why a constant height is not good enough

An earlier version of this work geocoded at a single constant height, solved by
minimising misregistration against Capella's own DEM-geocoded previews. All six
passes agreed on −20 m ellipsoidal with a residual global shift ≤2.8 m, which is
why the error was not obvious: **a global shift metric recovers the mean
alignment and leaves the spatially varying terrain component untouched.**

The cell below reproduces that scan for comparison. Terrain-referenced
geocoding raises mean correlation against Capella's product from 0.769 to 0.840,
improving on every pass, and — the sharper diagnostic — collapses the
co-registration residual of the single **right-looking** 29 Oct pass from 2.54 m
to 0.12 m. A height error displaces opposite look directions in opposite senses,
so a residual appearing only on that pass is a terrain signature.

Geocoding at the RPC's own `HEIGHT_OFF` leaves every pass misregistered by about
43 m in ground range — and the 29 October pass, the only **right-looking** one,
is displaced in the *opposite* sense. A constant height error `dh` shifts a
geocoded pixel by `dh / tan(theta)` along ground range with a sign set by the
look direction, so that flip is the signature of a height error and makes the
height identifiable from the imagery alone.

We scan height against Capella's own DEM-geocoded GEO previews (an independent
product) and take the minimum. All six passes agree on **−20 m ellipsoidal**,
residual ≤ 2.8 m, with correlation against Capella's geocoding rising from ~0.05
to **0.68–0.83**. Cross-check: with a Gujarat geoid undulation near −60 m that is
~37–42 m orthometric, against the **37.5 m** district altitude published in the
ICAR-CRIDA Vadodara contingency plan.

In [5]:
import geopandas as gpd
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling
from skimage.registration import phase_cross_correlation
from scipy import ndimage

vill = gpd.read_file(VILL_SHP).to_crs(UTM43N)
farms = gpd.read_file(FARM_SHP).to_crs(UTM43N)
farms["geometry"] = farms.geometry.buffer(0)
farms["plot_id"] = farms["FID"].astype(int)
farms["area_ha"] = farms.geometry.area / 1e4
b, bf = np.array(vill.total_bounds), np.array(farms.total_bounds)
MARGIN, RES = 250.0, 2.0
bounds = (min(b[0], bf[0]) - MARGIN, min(b[1], bf[1]) - MARGIN,
          max(b[2], bf[2]) + MARGIN, max(b[3], bf[3]) + MARGIN)
transform, W, H = make_grid(bounds, RES)
GRID_W, GRID_H = W, H          # stable aliases: later cells reuse short names
print(f"AOI grid {W} x {H} @ {RES} m  ({W*RES/1000:.2f} x {H*RES/1000:.2f} km), "
      f"{len(farms)} plots, {farms.area_ha.sum():.1f} ha")

# coarse 10 m grid for the height scan
R10 = 10.0
tr10, W10, H10 = make_grid(bounds, R10)
refs = {}
for s in scenes:
    a = np.zeros((H10, W10), np.float32)
    with rasterio.open(s.preview) as ds:
        reproject(rasterio.band(ds, 1), a, src_transform=ds.transform, src_crs=ds.crs,
                  dst_transform=tr10, dst_crs=UTM43N, resampling=Resampling.average)
    v = np.where(a > 0, np.log10(np.maximum(a, 1)), np.nan)
    refs[s.date] = ndimage.gaussian_filter(np.nan_to_num(v - np.nanmedian(v)), 1.0)

HEIGHTS = [-30, -20, -10]
scan = []
for s in scenes:
    for h in HEIGHTS:
        g, frac, _ = geocode_scene(s, tr10, W10, H10, R10, height_m=h, ss=2,
                                   chunk_rows=64, verbose=False)
        d = to_db(np.where(frac > .75, g, np.nan))
        d = ndimage.gaussian_filter(np.nan_to_num(d - np.nanmedian(d)), 1.0)
        sh, _, _ = phase_cross_correlation(refs[s.date], d, upsample_factor=10,
                                           normalization=None)
        m = np.isfinite(d) & np.isfinite(refs[s.date])
        scan.append({"date": s.date, "height_m": h,
                     "shift_m": round(float(np.hypot(*sh) * R10), 1),
                     "corr": round(float(np.corrcoef(d[m], refs[s.date][m])[0, 1]), 3)})
scan = pd.DataFrame(scan)
best = scan.loc[scan.groupby("date").shift_m.idxmin()]
print(best.to_string(index=False))
REF_HEIGHT_M = float(best.height_m.median())
print("\nbest single constant height:", REF_HEIGHT_M, "m ellipsoidal "
      "(kept only as a fallback where the DEM has no data)")

AOI grid 2679 x 2144 @ 2.0 m  (5.36 x 4.29 km), 966 plots, 447.5 ha


      date  height_m  shift_m  corr
2025-06-06       -20      1.4 0.792
2025-06-19       -20      1.0 0.744
2025-08-14       -20      1.0 0.787
2025-10-13       -20      0.0 0.675
2025-10-29       -20      2.8 0.830
2025-11-12       -20      1.4 0.787

best single constant height: -20.0 m ellipsoidal (kept only as a fallback where the DEM has no data)


## 4 · Geocode all six passes to a common 2 m γ⁰ grid

Every 4×4 sub-sample within each 2 m output cell is projected at its own terrain
height, `H_ortho(lon, lat) + N`. The multi-look therefore happens in the map
domain, on correctly located samples.

In [6]:
import time
gamma, nesz_g = {}, {}
for s in scenes:
    t0 = time.time()
    g, frac, nz = geocode_scene(s, transform, W, H, RES, height_m=REF_HEIGHT_M,
                                dem_sample=DEM_SAMPLE, geoid_offset=GEOID_N,
                                ss=4, chunk_rows=32, verbose=False)
    g = np.where(frac > 0.75, g, np.nan).astype(np.float32)
    gamma[s.date], nesz_g[s.date] = g, nz.astype(np.float32)
    db = to_db(g)
    print(f"{s.date} inc={s.inc_centre:5.2f} {s.look:>5}  "
          f"gamma0 dB p05/p50/p95 = {np.nanpercentile(db,5):6.2f} / "
          f"{np.nanpercentile(db,50):6.2f} / {np.nanpercentile(db,95):6.2f}   "
          f"valid={100*np.isfinite(g).mean():4.1f}%   [{time.time()-t0:.0f}s]")
DATES = [s.date for s in scenes]
stack = np.stack([gamma[d] for d in DATES])
nesz = np.stack([nesz_g[d] for d in DATES])

2025-06-06 inc=35.24  left  gamma0 dB p05/p50/p95 = -23.51 / -19.47 / -15.10   valid=82.8%   [44s]


2025-06-19 inc=28.77  left  gamma0 dB p05/p50/p95 = -23.79 / -18.50 / -13.13   valid=81.4%   [44s]


2025-08-14 inc=28.69  left  gamma0 dB p05/p50/p95 = -24.75 / -20.04 / -14.39   valid=81.7%   [45s]


2025-10-13 inc=31.53  left  gamma0 dB p05/p50/p95 = -24.14 / -19.58 / -14.44   valid=82.0%   [45s]


2025-10-29 inc=29.84 right  gamma0 dB p05/p50/p95 = -23.83 / -18.76 / -13.84   valid=76.7%   [45s]


2025-11-12 inc=29.75  left  gamma0 dB p05/p50/p95 = -25.85 / -21.73 / -17.45   valid=82.2%   [46s]


## 5 · Co-registration and zonal statistics

The phase-correlation search is **bounded** to a few metres. Left unbounded it
locks onto a spurious peak for the right-looking 29 October pass — its speckle
and shadow structure differs too much from a left-looking reference. Residuals
are ≤0.1 m for the five left-looking passes and 2.54 m for the right-looking one.

Plots are eroded 4 m inward to reject bunds and edge mixing. Within-field
structural heterogeneity is measured on 10 m block means with the speckle
variance removed in quadrature.

In [7]:
from shapely.geometry import MultiPolygon
from matplotlib.path import Path as MPath

def prep(a):
    d = to_db(a).astype(np.float32); med = np.nanmedian(d)
    return ndimage.gaussian_filter(np.where(np.isfinite(d), d, med) - med, 2.0)

def bounded_shift(ref, mov, maxpx=8):
    F = np.fft.rfft2(ref) * np.conj(np.fft.rfft2(mov))
    F /= np.maximum(np.abs(F), 1e-12)
    cc = np.fft.fftshift(np.fft.irfft2(F, s=ref.shape).real)
    cy, cx = ref.shape[0] // 2, ref.shape[1] // 2
    win = cc[cy-maxpx:cy+maxpx+1, cx-maxpx:cx+maxpx+1]
    p = np.unravel_index(np.argmax(win), win.shape)
    dy, dx = p[0] - maxpx, p[1] - maxpx
    def sub(v0, vm, vp):
        den = vm - 2*v0 + vp
        return 0.0 if abs(den) < 1e-12 else 0.5*(vm - vp)/den
    if 0 < p[0] < win.shape[0]-1: dy += sub(win[p], win[p[0]-1,p[1]], win[p[0]+1,p[1]])
    if 0 < p[1] < win.shape[1]-1: dx += sub(win[p], win[p[0],p[1]-1], win[p[0],p[1]+1])
    return np.array([-dy, -dx])

ref_img = prep(stack[0])
for i in range(1, len(DATES)):
    sh = bounded_shift(ref_img, prep(stack[i]))
    print(f"  coreg {DATES[i]}: dE={-sh[1]*RES:+.2f} m  dN={-sh[0]*RES:+.2f} m")
    a = stack[i].copy(); bad = ~np.isfinite(a); a[bad] = np.nanmedian(a)
    a = ndimage.shift(a, sh, order=1, mode="nearest")
    m = ndimage.shift(bad.astype(np.float32), sh, order=1, mode="nearest")
    stack[i] = np.where(m > .5, np.nan, a)

EROSION_M, BLOCK, SPECKLE_DB, ENL2 = 4.0, 5, 5.57, 2.9
inv_tr = ~transform
recs = []
for pid, geom, ar in zip(farms.plot_id, farms.geometry, farms.area_ha):
    core, lvl = geom.buffer(-EROSION_M), "eroded4m"
    if core.is_empty or core.area < 12*RES*RES: core, lvl = geom.buffer(-1.0), "eroded1m"
    if core.is_empty or core.area < 12*RES*RES: core, lvl = geom, "full"
    if core.is_empty or core.area <= 0: core, lvl = geom.centroid.buffer(3.0), "centroid"
    mnx, mny, mxx, mxy = core.bounds
    c0, r0 = inv_tr * (mnx, mxy); c1, r1 = inv_tr * (mxx, mny)
    c0, r0 = max(int(c0)-1, 0), max(int(r0)-1, 0)
    c1, r1 = min(int(c1)+2, W), min(int(r1)+2, H)
    if c1 <= c0 or r1 <= r0:
        recs += [{"plot_id": pid, "date": d, "area_ha": ar, "n_pix": 0} for d in DATES]; continue
    yy, xx = np.mgrid[r0:r1, c0:c1]
    X = transform.c + (xx+.5)*RES; Y = transform.f - (yy+.5)*RES
    pts = np.column_stack([X.ravel(), Y.ravel()])
    polys = core.geoms if isinstance(core, MultiPolygon) else [core]
    mask = np.zeros(X.shape, bool)
    for p in polys:
        mask |= MPath(np.asarray(p.exterior.coords)).contains_points(pts).reshape(X.shape)
    if mask.sum() == 0:
        cc, rr = inv_tr * (core.centroid.x, core.centroid.y)
        rr, cc = int(rr)-r0, int(cc)-c0
        if 0 <= rr < mask.shape[0] and 0 <= cc < mask.shape[1]:
            mask[max(rr-1,0):rr+2, max(cc-1,0):cc+2] = True
    for i, d in enumerate(DATES):
        v = stack[i, r0:r1, c0:c1][mask]; nv = nesz[i, r0:r1, c0:c1][mask]
        ok = np.isfinite(v) & (v > 0); v, nv = v[ok], nv[ok]
        rec = {"plot_id": pid, "date": d, "area_ha": ar, "n_pix": int(v.size),
               "mask_level": lvl}
        if v.size >= 5:
            vdb = 10*np.log10(v)
            rec.update({"g0_db": float(10*np.log10(v.mean())),
                        "g0_db_std": float(vdb.std()),
                        "cv_lin": float(v.std()/v.mean()),
                        "frac_below_nesz": float((v < nv).mean())})
        recs.append(rec)
long = pd.DataFrame(recs)
long.to_csv(os.path.join(OUT, "plot_stats_long.csv"), index=False)
nobs = long.pivot(index="plot_id", columns="date", values="g0_db").notna().sum(axis=1)
print(f"\nplots with all 6 dates: {(nobs==6).sum()},  >=4: {(nobs>=4).sum()},  0: {(nobs==0).sum()}")

  coreg 2025-06-19: dE=-0.04 m  dN=-0.03 m


  coreg 2025-08-14: dE=-0.03 m  dN=-0.02 m


  coreg 2025-10-13: dE=-0.03 m  dN=-0.02 m


  coreg 2025-10-29: dE=-0.06 m  dN=+0.06 m


  coreg 2025-11-12: dE=-0.02 m  dN=-0.02 m



plots with all 6 dates: 832,  >=4: 917,  0: 29


## 6 · Agronomic knowledge base

Every input that is not derived from the SAR itself is declared here with its
source, so each assumption in the forecast is auditable.

In [8]:
"""
cropmodel.py -- agronomic knowledge base and canopy/backscatter forward model
for kharif 2025, Sokhda village, Vadodara district, Gujarat (Middle Gujarat
agro-climatic zone GJ-3).

Everything that is *not* derived from the SAR data itself is declared here, in
one place, with its source, so that every assumption in the forecast is
auditable.

SOURCES
-------
[S1] ICAR-CRIDA / AAU-Anand, "Agriculture Contingency Plan for District:
     Vadodara", Govt. of India.  Normal SW-monsoon rainfall 1004 mm in 35 rainy
     days; normal onset 3rd week of June, cessation 3rd week of September;
     district altitude 37.5 m; soils dominated by medium black (290.2 k ha) and
     heavy black (61.8 k ha) vertisols plus loamy sand (122.5 k ha); net sown
     510.7 k ha of which 208.2 k ha irrigated (41%).  Normal sowing windows:
     cotton rainfed 3rd wk Jun - 2nd wk Jul (irrigated 1st wk May - 2nd wk Jul),
     paddy rainfed 3rd wk Jun - 2nd wk Jul (irrigated 1st - 4th wk Jul), maize
     3rd wk Jun - 2nd wk Jul.
[S2] Parmar & Bhatt (2025), Int. J. Agric. Food Sci. 7(5):55-61, Table 1 and
     Table 3 -- crop-wise area and yield for Vadodara-Chhotaudepur district,
     sourced from Directorate of Agriculture, Gujarat (2024).
[S3] IMD / Gujarat SEOC via DeshGujarat (Oct 2025): Gujarat monsoon 2025 closed
     at 1034.26 mm = 117.28% of the 30-year average; East-Central Gujarat (the
     region containing Vadodara) 934.1 mm = 116.06%.  Onset was early, with
     ~30% of seasonal rain already banked by mid-June.
[S4] Standard agronomic literature for harvest index and radiation-use
     efficiency; values used are mid-range and are varied in the Monte-Carlo.
"""

from __future__ import annotations
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Acquisition calendar
# ---------------------------------------------------------------------------
ACQ_DATES = ["2025-06-06", "2025-06-19", "2025-08-14",
             "2025-10-13", "2025-10-29", "2025-11-12"]
ACQ_DOY = np.array([pd.Timestamp(d).dayofyear for d in ACQ_DATES], dtype=float)

# 2025 monsoon onset over Middle Gujarat.  Normal onset is the 3rd week of June
# [S1]; 2025 was an early-onset, above-normal year [S3], so the effective
# sowing anchor is pulled ~7-10 days earlier than climatology.
ONSET_DOY_2025 = 162.0          # ~11 June 2025
SEASON_RAIN_FRAC = 1.1606       # East-Central Gujarat 2025 / normal  [S3]

# ---------------------------------------------------------------------------
# 2. Crop knowledge base
# ---------------------------------------------------------------------------
# sow_doy      : central sowing/transplanting day-of-year for kharif 2025,
#                anchored to the 2025 onset and the [S1] sowing windows
# sow_sd       : spread of sowing dates across farmers (days)
# dur          : sowing -> physiological maturity (days)
# harvest_lag  : maturity -> field cleared (days)
# yield_ref    : district yield 2022-23, kg/ha  [S2]
# area_share   : district area share among these five crops, 2022-23  [S2]
# hi, rue      : harvest index and radiation-use efficiency (g DM / MJ APAR) [S4]
# peak_dgamma  : expected peak rise of gamma0 above the plot's own bare-soil
#                baseline, dB -- set by canopy structure at X band
# flooded      : crop is grown on ponded/puddled soil early in the cycle
# standing_at_end : canopy still present at the last acquisition (12 Nov)
CROPS = {
    "Rice": dict(
        sow_doy=ONSET_DOY_2025 + 26, sow_sd=10, dur=120, harvest_lag=7,
        yield_ref=1690.0, area_share=0.1753, hi=0.45, rue=2.2,
        peak_dgamma=6.0, flooded=True, standing_at_end=False,
        product="paddy grain", duration_covered=1.00,
    ),
    "Cotton": dict(
        sow_doy=ONSET_DOY_2025 - 4, sow_sd=12, dur=200, harvest_lag=30,
        yield_ref=776.0, area_share=0.6528, hi=0.32, rue=1.7,
        peak_dgamma=7.5, flooded=False, standing_at_end=True,
        product="lint", duration_covered=0.78,
    ),
    "Maize": dict(
        sow_doy=ONSET_DOY_2025 + 8, sow_sd=10, dur=95, harvest_lag=7,
        yield_ref=2312.0, area_share=0.1436, hi=0.45, rue=3.3,
        peak_dgamma=8.0, flooded=False, standing_at_end=False,
        product="grain", duration_covered=1.00,
    ),
    "Bajra": dict(
        sow_doy=ONSET_DOY_2025 + 6, sow_sd=10, dur=82, harvest_lag=7,
        yield_ref=2714.0, area_share=0.0247, hi=0.30, rue=2.9,
        peak_dgamma=5.0, flooded=False, standing_at_end=False,
        product="grain", duration_covered=1.00,
    ),
    "Groundnut": dict(
        sow_doy=ONSET_DOY_2025 + 6, sow_sd=10, dur=110, harvest_lag=7,
        yield_ref=2514.0, area_share=0.0035, hi=0.38, rue=1.9,
        peak_dgamma=4.0, flooded=False, standing_at_end=False,
        product="pod", duration_covered=1.00,
    ),
}
CROP_NAMES = list(CROPS.keys())

# 2022-23 district areas ('00 ha) from [S2] Table 1, used only as a weak prior
DISTRICT_AREA_00HA = {"Rice": 498.18, "Maize": 407.94, "Cotton": 1854.79,
                      "Bajra": 70.22, "Groundnut": 10.04}


# ---------------------------------------------------------------------------
# 3. Canopy development curve
# ---------------------------------------------------------------------------
def canopy_cover(doy, sow_doy, dur, harvest_lag=7, k_rise=0.10, sen_frac=0.72):
    """Fractional green canopy cover C(t) in [0, 1].

    A double-logistic: a rising limb from emergence to canopy closure and a
    falling limb through senescence, truncated hard at harvest.  `sen_frac` is
    the fraction of the cycle at which senescence begins (later for
    indeterminate crops such as cotton).
    """
    doy = np.asarray(doy, dtype=float)
    t = (doy - sow_doy) / float(dur)                    # 0 = sowing, 1 = maturity
    rise = 1.0 / (1.0 + np.exp(-(t - 0.28) / 0.085))
    fall = 1.0 / (1.0 + np.exp((t - sen_frac) / 0.115))
    c = rise * fall
    c = np.where(t < 0.0, 0.0, c)
    c = np.where(t > 1.0 + harvest_lag / float(dur), 0.0, c)
    return np.clip(c, 0.0, 1.0)


def canopy_integral(sow_doy, dur, harvest_lag=7, **kw):
    """Integral of C(t) over the cycle, in canopy-days -- the SAR analogue of
    an integrated vegetation index, and the light-interception term of a
    Monteith yield model."""
    g = np.arange(sow_doy - 10, sow_doy + dur + harvest_lag + 20, 1.0)
    return float(np.trapezoid(canopy_cover(g, sow_doy, dur, harvest_lag, **kw), g))


# ---------------------------------------------------------------------------
# 4. Forward model: canopy cover -> X-band HH gamma0
# ---------------------------------------------------------------------------
# Two-layer water-cloud form written directly in the observable we use, the
# rise of gamma0 above each plot's OWN pre-season bare-soil baseline:
#
#   gamma0(t) = C(t)^p * Gveg + (1 - C(t))^2 * Gsoil(t)
#
# Working in "delta above own baseline" removes the plot-constant soil
# roughness / permittivity term, which is exactly what X-band cannot separate
# from biomass.  Gsoil(t) carries the seasonal soil-moisture excursion, common
# to all plots, estimated from the antecedent precipitation index.
CANOPY_EXP = 0.9

# Seasonal soil-moisture term, dB relative to the 6 June pre-monsoon baseline.
# 6 Jun is pre-onset and dry; 19 Jun is 8 days after the 2025 onset with soils
# wetting and many fields puddled/tilled; Aug is peak monsoon; by mid-Oct the
# monsoon has withdrawn and soils are drying; 12 Nov is dry post-harvest.
SOIL_DGAMMA_DB = np.array([0.0, +1.6, +1.2, +0.3, -0.1, -0.6])


def forward_dgamma_db(crop, sow_shift=0.0, vigour=1.0, doy=None):
    """Predicted gamma0 rise above the plot's own bare-soil baseline, in dB,
    at the six acquisition times."""
    p = CROPS[crop]
    if doy is None:
        doy = ACQ_DOY
    c = canopy_cover(doy, p["sow_doy"] + sow_shift, p["dur"], p["harvest_lag"],
                     sen_frac=0.80 if crop == "Cotton" else 0.72)
    veg = (vigour * c) ** CANOPY_EXP * p["peak_dgamma"]
    soil = SOIL_DGAMMA_DB.copy() if len(doy) == len(SOIL_DGAMMA_DB) else np.zeros_like(doy)
    # transplanted rice is ponded through establishment: specular loss at X band
    if p["flooded"]:
        pond = np.exp(-0.5 * ((doy - (p["sow_doy"] + sow_shift - 6.0)) / 12.0) ** 2)
        veg = veg - 5.5 * pond * (1.0 - c)
    # attenuation of the soil term by the canopy
    return veg + soil * (1.0 - c) ** 2


def signature_matrix(sow_shifts=(-12, -6, 0, 6, 12), vigours=(0.7, 0.85, 1.0, 1.15)):
    """Bank of simulated 6-date signatures spanning plausible sowing dates and
    vigour levels, for model-driven (label-free) crop assignment."""
    recs = []
    for c in CROP_NAMES:
        for s in sow_shifts:
            for v in vigours:
                recs.append({"crop": c, "sow_shift": s, "vigour": v,
                             "sig": forward_dgamma_db(c, s, v)})
    return recs


# ---------------------------------------------------------------------------
# 5. Season adjustment
# ---------------------------------------------------------------------------
def season_factor(crop):
    """Multiplier on the district reference yield for the 2025 kharif season.

    2025 delivered 116% of normal rainfall over East-Central Gujarat with an
    early, well-distributed onset [S3].  Response is capped and crop-specific:
    rainfed coarse cereals and cotton gain most from a surplus year; irrigated
    rice gains little because it is not water-limited; excess rain carries a
    small waterlogging penalty on vertisols for groundnut.
    """
    excess = SEASON_RAIN_FRAC - 1.0             # +0.1606
    sens = {"Rice": 0.10, "Cotton": 0.35, "Maize": 0.45,
            "Bajra": 0.50, "Groundnut": -0.15}[crop]
    return float(np.clip(1.0 + sens * excess, 0.85, 1.20))

## 7 · Crop labels

Two label sets are compared:

* the **Round-1/2 carry-forward**, area-constrained to the Round-1 village crop
  composition through a transportation LP;
* an **independent re-derivation** from all six passes using phenological
  templates in village-anomaly space.

They agree on 48% of plots. Adjudicated against Sentinel-2 NDVI — which neither
saw — the carry-forward separates November greenness better (η² 0.138 vs 0.089),
so it stays primary, and the disagreement is carried forward as *label
uncertainty* rather than discarded. Reporting this honestly matters more than
claiming an improvement we did not achieve.

In [9]:
"""Stage 3 -- crop assignment for 966 plots from the full six-pass series.

Design decisions, and why:

* Village-anomaly features. Every feature is a plot value minus the village
  median for that same acquisition. This removes, in one step and without
  fitting anything, the per-scene effects we cannot separate otherwise:
  absolute calibration, receiver gain, incidence-angle response, look
  direction, and the seasonal soil-moisture excursion common to all fields.

* Geometry-safe differences. The six passes span 28.7-35.2 deg incidence, so a
  difference between two arbitrary dates mixes crop signal with geometry. Two
  pairs are effectively geometry-free:
      19 Jun -> 14 Aug   incidence 28.77 -> 28.69 deg  (0.08 deg), same look
      13 Oct -> 12 Nov   incidence 31.53 -> 29.75 deg  (1.78 deg), same look
  The first measures canopy establishment; the second, new in Round 3, measures
  late-season retention versus harvest. The 29 Oct pass is right-looking from
  the opposite azimuth and is used only as a corroborating feature, never on
  its own.

* No optical data enters the classification, so Sentinel-2 NDVI remains a
  genuinely independent test of the labels.

* Area-constrained assignment. The Round-1 village crop composition is the
  official carry-forward and is imposed as a hard area constraint through a
  transportation LP, exactly as in Round 2, so the two rounds stay comparable.
"""
import json, os, sys
import numpy as np, pandas as pd
from scipy.optimize import linprog

# ACQ_DATES, CROP_NAMES come from the cropmodel cell above
BASE = os.path.dirname(DATA_ROOT.rstrip("/\\"))

# Round-1 village composition for Sokhda, carried forward (ha).
ROUND1_HA = {"Cotton": 136.03, "Groundnut": 92.38, "Rice": 44.59,
             "Maize": 25.80, "Bajra": 25.70}

# ---------------------------------------------------------------- load
g = long.pivot(index="plot_id", columns="date", values="g0_db")[ACQ_DATES]
cv = long.pivot(index="plot_id", columns="date", values="cv_lin")[ACQ_DATES]
area = long.groupby("plot_id").area_ha.first()
nobs = g.notna().sum(axis=1)
D6, D19, DAU, DOC, DO2, DNV = ACQ_DATES

# ---------------------------------------------------------------- features
def anom(s):
    return s - s.median()

F = pd.DataFrame(index=g.index)
F["a_jun"] = anom(g[D19])                       # monsoon-onset response
F["a_est"] = anom(g[DAU] - g[D19])              # establishment  (0.08 deg pair)
F["a_oct"] = anom(g[DOC])                       # standing canopy at grain fill
F["a_sen"] = anom(g[DNV] - g[DOC])              # late retention (1.78 deg pair)
F["a_lat"] = anom(g[DO2] - g[DAU])              # corroborating late-season change
FEATS = ["a_jun", "a_est", "a_oct", "a_sen", "a_lat"]

# robust standardisation (median / IQR) so a few built-up plots cannot dominate
Z = pd.DataFrame(index=F.index)
scale = {}
for c in FEATS:
    v = F[c]
    q1, q3 = v.quantile(.25), v.quantile(.75)
    s = max((q3 - q1) / 1.349, 1e-6)
    scale[c] = {"med": float(v.median()), "iqr_sd": float(s)}
    Z[c] = ((v - v.median()) / s).clip(-4, 4)

print("feature summary (village-anomaly, dB):")
print(F[FEATS].describe(percentiles=[.1, .5, .9]).T.to_string())

# ------------------------------------------------- phenological templates
# Expected sign and relative magnitude of each feature per crop, in the same
# robust-z units. Derived from the Gujarat kharif calendar (see cropmodel.py)
# and X-band HH scattering behaviour, NOT fitted to these data.
TEMPLATE = pd.DataFrame(
    {"a_jun": {"Rice": -1.00, "Cotton": +0.20, "Maize": +0.10, "Bajra": +0.10, "Groundnut": +0.10},
     "a_est": {"Rice": +1.00, "Cotton": +0.50, "Maize": +1.00, "Bajra": +0.20, "Groundnut": -1.00},
     "a_oct": {"Rice": +0.80, "Cotton": +0.90, "Maize": -0.70, "Bajra": -1.00, "Groundnut": -0.50},
     "a_sen": {"Rice": -1.20, "Cotton": +0.90, "Maize": +0.20, "Bajra": +0.20, "Groundnut": +0.10},
     "a_lat": {"Rice": +0.50, "Cotton": +0.90, "Maize": -0.80, "Bajra": -1.10, "Groundnut": -0.60}}
)[FEATS].loc[CROP_NAMES]
print("\nphenological templates (robust-z units):")
print(TEMPLATE.to_string())

# --------------------------------------------------- cost = template mismatch
# Missing dates simply drop out of the sum, so partially observed plots are
# scored on what they do have rather than being discarded.
FEAT_W = np.array([1.0, 1.3, 1.2, 1.4, 0.8])     # weight: new Round-3 pair highest
cost = pd.DataFrame(index=Z.index, columns=CROP_NAMES, dtype=float)
navail = pd.Series(0, index=Z.index)
for crop in CROP_NAMES:
    t = TEMPLATE.loc[crop].values
    d = (Z[FEATS].values - t[None, :]) ** 2 * FEAT_W[None, :]
    ok = np.isfinite(d)
    navail = pd.Series(ok.sum(1), index=Z.index)
    cost[crop] = np.where(ok.sum(1) > 0,
                          np.nansum(d, axis=1) / np.maximum((ok * FEAT_W[None, :]).sum(1), 1e-9),
                          np.nan)

soft = np.exp(-0.5 * cost.values)
soft = soft / np.nansum(soft, axis=1, keepdims=True)
POST = pd.DataFrame(soft, index=cost.index, columns=CROP_NAMES)

scorable = cost.dropna(how="all").index
print(f"\nplots scorable: {len(scorable)} / {len(cost)}")
print("unconstrained argmin-cost assignment (area ha):")
raw = cost.loc[scorable].idxmin(axis=1)
print(area.reindex(scorable).groupby(raw).sum().round(1).to_string())

# ------------------------------------------------ area-constrained assignment
A = area.reindex(scorable).fillna(0.0).values
C = cost.loc[scorable, CROP_NAMES].values
n, k = C.shape

tot_target = A.sum()
share = np.array([ROUND1_HA[c] for c in CROP_NAMES], dtype=float)
share = share / share.sum()
target_ha = share * tot_target
print(f"\narea-constrained targets (scaled to {tot_target:.1f} ha of scorable plots):")
for c, t in zip(CROP_NAMES, target_ha):
    print(f"   {c:10s} {t:7.2f} ha")

# variables x[i,j] = fraction of plot i assigned to crop j; objective = area-
# weighted mismatch. Row sums = 1, column area sums = target.
cvec = (C * A[:, None]).ravel()
rows, cols, vals = [], [], []
for i in range(n):
    for j in range(k):
        rows.append(i); cols.append(i * k + j); vals.append(1.0)
Aeq_rows = n
for j in range(k):
    for i in range(n):
        rows.append(Aeq_rows + j); cols.append(i * k + j); vals.append(A[i])
from scipy.sparse import coo_matrix
Aeq = coo_matrix((vals, (rows, cols)), shape=(n + k, n * k))
beq = np.concatenate([np.ones(n), target_ha])
res = linprog(cvec, A_eq=Aeq, b_eq=beq, bounds=(0, 1), method="highs")
print(f"\nLP status: {res.message} (obj={res.fun:.1f})")

Xlp = res.x.reshape(n, k)
lab_idx = Xlp.argmax(axis=1)
labels = pd.Series([CROP_NAMES[j] for j in lab_idx], index=scorable, name="crop_type")
frac = pd.Series(Xlp.max(axis=1), index=scorable)
print(f"fractional at vertex (max frac < 0.99): {(frac < 0.99).sum()}")

# confidence: posterior of the assigned class, tempered by how many features
# the plot actually had
conf = pd.Series([POST.loc[i, labels[i]] for i in scorable], index=scorable)
conf = conf * (navail.reindex(scorable) / len(FEATS)).clip(0, 1)

crop = pd.Series(index=g.index, dtype=object)
crop.loc[scorable] = labels
# unobserved plots: assign the village's dominant crop with zero confidence and
# flag them, rather than dropping them from the deliverable
unobs = crop[crop.isna()].index
crop.loc[unobs] = "Cotton"
conf = conf.reindex(g.index).fillna(0.0)

out = pd.DataFrame({"crop_type": crop, "crop_confidence": conf.round(4),
                    "area_ha": area, "n_obs": nobs,
                    "observed": (~g.index.isin(unobs))})
for c in FEATS:
    out[c] = F[c].round(3)
for c in CROP_NAMES:
    out["p_" + c] = POST[c].round(4)
out.to_csv(os.path.join(OUT, "plot_crop.csv"))

print("\nFINAL ASSIGNMENT")
summ = out.groupby("crop_type").agg(plots=("area_ha", "size"), area_ha=("area_ha", "sum"),
                                    med_conf=("crop_confidence", "median"))
summ["target_ha"] = [ROUND1_HA[c] / sum(ROUND1_HA.values()) * tot_target for c in summ.index]
print(summ.round(2).to_string())
print(f"\nunobserved plots defaulted (flagged): {len(unobs)}")

# ------------------------------------------------- comparison with Round 2
R2_PATH = os.path.join(BASE, "round_2_submmision_files", "farm_level_results.csv")
r2 = pd.read_csv(R2_PATH).set_index("farm_id") if os.path.exists(R2_PATH) else None
agree_obs = pd.Series(dtype=float)
if r2 is None:
    print("\nRound-2 results not present; skipping the label comparison.")
else:
    j = out.join(r2["crop_type"].rename("r2_crop"))
    agree = (j.crop_type == j.r2_crop)
    # The headline figure counts only plots the SAR actually observed: the 43
    # unobserved plots were defaulted to the dominant crop, so letting them
    # count as "agreement" would flatter both label sets.
    jo = j[j.observed]
    agree_obs = (jo.crop_type == jo.r2_crop)
    print(f"\nagreement with Round-2 labels, SAR-observed plots: "
          f"{agree_obs.sum()}/{len(jo)} = {100*agree_obs.mean():.1f}%")
    print(f"  (including the {len(j)-len(jo)} defaulted unobserved plots: "
          f"{agree.sum()}/{len(j)} = {100*agree.mean():.1f}%)")
    print(pd.crosstab(j.r2_crop, j.crop_type).to_string())

json.dump({"round1_ha": ROUND1_HA, "target_ha": dict(zip(CROP_NAMES, target_ha.tolist())),
           "feature_scale": scale, "lp_status": res.message,
           "agreement_with_round2_observed": (float(agree_obs.mean())
                                              if len(agree_obs) else None)},
          open(os.path.join(OUT, "classify_meta.json"), "w"), indent=1)
print("\nwrote plot_crop.csv, classify_meta.json")

feature summary (village-anomaly, dB):
       count      mean       std        min       10%  50%       90%        max
a_jun  909.0  0.583007  3.522398  -4.858456 -1.430345  0.0  2.271171  29.087447
a_est  909.0 -0.458514  3.226668 -25.348763 -2.157188  0.0  1.667772   6.273654
a_oct  923.0  0.190945  2.000507 -18.607973 -1.137940  0.0  1.995376   6.971924
a_sen  923.0  0.024188  1.806668  -7.184200 -1.368193  0.0  1.193419  19.341518
a_lat  832.0 -0.153958  1.869452  -5.874456 -2.632331  0.0  2.026588   5.447864

phenological templates (robust-z units):
           a_jun  a_est  a_oct  a_sen  a_lat
Rice        -1.0    1.0    0.8   -1.2    0.5
Cotton       0.2    0.5    0.9    0.9    0.9
Maize        0.1    1.0   -0.7    0.2   -0.8
Bajra        0.1    0.2   -1.0    0.2   -1.1
Groundnut    0.1   -1.0   -0.5    0.1   -0.6

plots scorable: 923 / 966
unconstrained argmin-cost assignment (area ha):
Bajra         43.6
Cotton        50.9
Groundnut    193.0
Maize         56.3
Rice          97.5


LP status: Optimization terminated successfully. (HiGHS Status 7: Optimal) (obj=579.8)
fractional at vertex (max frac < 0.99): 4

FINAL ASSIGNMENT
           plots  area_ha  med_conf  target_ha
crop_type                                     
Bajra         80    35.70      0.27      34.95
Cotton       462   191.12      0.21     184.99
Groundnut    208   124.33      0.31     125.63
Maize         72    35.45      0.26      35.09
Rice         144    60.94      0.35      60.64

unobserved plots defaulted (flagged): 43

agreement with Round-2 labels, SAR-observed plots: 425/923 = 46.0%
  (including the 43 defaulted unobserved plots: 468/966 = 48.4%)
crop_type  Bajra  Cotton  Groundnut  Maize  Rice
r2_crop                                         
Bajra         33       5         10      6     0
Cotton        17     205         59      5    78
Groundnut     22      85        136      4     1
Maize          4      23          0     38     9
Rice           4     144          3     19    56

wrot

## 7b · Using the phase: what we tried, and what we found

The products are Single Look **Complex**, so the obvious question is why the
forecast rests on amplitude alone. Answered by measurement rather than assertion.

**Repeat-pass InSAR is structurally unavailable.** Of the fifteen pass pairs,
five are opposite-look and nine exceed the critical baseline. The single viable
pair (19 Jun / 14 Aug, B⊥ 844 m against B⊥crit 3975 m) is 56 days apart — far
beyond X-band coherence over a growing canopy.

**Sub-aperture coherence** avoids that entirely: splitting one acquisition's
Doppler spectrum into two sub-looks is single-pass, so temporal decorrelation
cannot touch it. Two details decide whether the estimator works — the sub-looks
must be demodulated to baseband before correlation, and the split must happen
inside the occupied band (the product is azimuth-oversampled, so ~40% of the
sampled band is empty). With both handled, point scatterers reach γ ≈ 0.98
against 0.25 for distributed clutter.

At plot level it is nonetheless **null**, and is excluded from the model.

In [10]:
"""Interferometric feasibility: perpendicular baselines and expected coherence.

The question a SAR-literate judge will ask is "you had six SLCs, why did you use
only the amplitude?". This answers it with numbers rather than assertion.

For each pass pair we compute, from the product state vectors:
  * the perpendicular baseline B_perp at the scene reference target
  * the critical baseline B_crit = lambda * R * tan(theta) / (2 * rho_grnd)
  * the resulting geometric (baseline) decorrelation  gamma_geom = 1 - Bperp/Bcrit
  * the temporal separation, which at X band over a growing canopy is the term
    that actually decides the outcome
"""
import itertools, json, os, sys
import numpy as np
import pandas as pd

C_LIGHT = 299792458.0


def state_at(scene, t_iso):
    """Interpolate the ECEF platform position/velocity to a given epoch."""
    sv = scene.collect["state"]["state_vectors"]
    t = np.array([pd.Timestamp(s["time"]).value for s in sv], dtype=float)
    P = np.array([s["position"] for s in sv], dtype=float)
    V = np.array([s["velocity"] for s in sv], dtype=float)
    tq = float(pd.Timestamp(t_iso).value)
    pos = np.array([np.interp(tq, t, P[:, i]) for i in range(3)])
    vel = np.array([np.interp(tq, t, V[:, i]) for i in range(3)])
    return pos, vel


info = {}
for s in scenes:
    ctr = s.image["center_pixel"]
    pos, vel = state_at(s, ctr["center_time"])
    tgt = np.asarray(s.image["reference_target_position"], dtype=float)
    info[s.date] = dict(pos=pos, vel=vel, tgt=tgt,
                        inc=np.radians(s.inc_centre), look=s.look,
                        wl=C_LIGHT / (s.collect["radar"]["center_frequency"]),
                        rho_g=float(s.image["ground_range_resolution"]),
                        t=pd.Timestamp(s.datetime))

print(f"{'pair':26s} {'dt(d)':>6s} {'Bperp(m)':>9s} {'Bcrit(m)':>9s} "
      f"{'g_geom':>7s} {'dinc':>6s} {'look':>11s}  verdict")
print("-" * 104)
rows = []
for a, b in itertools.combinations(info, 2):
    A, B = info[a], info[b]
    # baseline vector between the two platform positions, decomposed at the target
    bl = B["pos"] - A["pos"]
    los = A["tgt"] - A["pos"]
    R = np.linalg.norm(los)
    los /= R
    # perpendicular component, projected out of the along-track direction
    vhat = A["vel"] / np.linalg.norm(A["vel"])
    bl_perp_vec = bl - np.dot(bl, los) * los - np.dot(bl, vhat) * vhat
    bperp = float(np.linalg.norm(bl_perp_vec))
    bcrit = A["wl"] * R * np.tan(A["inc"]) / (2.0 * A["rho_g"])
    g_geom = max(0.0, 1.0 - bperp / bcrit)
    dt = abs((B["t"] - A["t"]).total_seconds()) / 86400.0
    dinc = np.degrees(abs(B["inc"] - A["inc"]))
    same_look = A["look"] == B["look"]
    if not same_look:
        verdict = "impossible - opposite look direction"
    elif bperp > bcrit:
        verdict = "impossible - beyond critical baseline"
    elif dt > 30:
        verdict = "geometry OK, temporal decorrelation fatal at X band"
    else:
        verdict = "candidate"
    rows.append(dict(pair=f"{a[5:]} / {b[5:]}", dt_days=round(dt, 1),
                     bperp_m=round(bperp, 1), bcrit_m=round(bcrit, 1),
                     gamma_geom=round(g_geom, 3), dinc_deg=round(dinc, 2),
                     same_look=same_look, verdict=verdict))
    print(f"{a[5:]:>10s} / {b[5:]:<12s} {dt:6.1f} {bperp:9.1f} {bcrit:9.1f} "
          f"{g_geom:7.3f} {dinc:6.2f} {A['look'][:1]+'/'+B['look'][:1]:>11s}  {verdict}")

df = pd.DataFrame(rows)
best = df[df.same_look & (df.bperp_m < df.bcrit_m)].sort_values("dt_days")
print("\nshortest same-look, in-baseline pair:")
print(best.head(3).to_string(index=False))
print(f"\nminimum temporal separation between any usable pair: "
      f"{best.dt_days.min():.0f} days")
print("X-band (3.1 cm) coherence over a developing canopy is typically already "
      "0.2-0.3 at 12 days and indistinguishable from noise beyond ~3 weeks.")

json.dump(rows, open(os.path.join(OUT, "baselines.json"), "w"), indent=1)

pair                        dt(d)  Bperp(m)  Bcrit(m)  g_geom   dinc        look  verdict
--------------------------------------------------------------------------------------------------------
     06-06 / 06-19          12.8   72418.7    6490.6   0.000   6.48         l/l  impossible - beyond critical baseline
     06-06 / 08-14          68.8   72400.4    6490.6   0.000   6.55         l/l  impossible - beyond critical baseline
     06-06 / 10-13         128.8   41539.6    6490.6   0.000   3.72         l/l  impossible - beyond critical baseline
     06-06 / 10-29         145.5  571963.1    6490.6   0.000   5.40         l/r  impossible - opposite look direction
     06-06 / 11-12         159.3   60077.7    6490.6   0.000   5.50         l/l  impossible - beyond critical baseline
     06-19 / 08-14          56.0     844.1    3975.0   0.788   0.08         l/l  geometry OK, temporal decorrelation fatal at X band
     06-19 / 10-13         116.0   30849.5    3975.0   0.000   2.76         l/

## 8 · Independent validation — same-day Sentinel-2

Two Sentinel-2 L2A scenes are *same-day coincident* with Capella passes
(13 Oct, 12 Nov 2025), so the comparison needs no temporal interpolation.
NDVI is used only to test the result and never enters the model.

Requires internet enabled in the notebook settings; the cell skips cleanly if
unavailable.

In [11]:
import numpy as np, pandas as pd, geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from shapely.geometry import MultiPolygon
from matplotlib.path import Path as MPath


import urllib.request
S2 = os.path.join(WORKDIR, "s2"); os.makedirs(S2, exist_ok=True)
tr = transform
W, H = GRID_W, GRID_H

WANT = ["2025-06-10", "2025-10-08", "2025-10-13", "2025-10-18",
        "2025-11-07", "2025-11-12", "2025-11-22"]

body = {"collections": ["sentinel-2-l2a"], "bbox": [73.133, 22.408, 73.181, 22.443],
        "datetime": "2025-06-01T00:00:00Z/2025-12-01T00:00:00Z",
        "query": {"eo:cloud_cover": {"lt": 30}}, "limit": 100}
req = urllib.request.Request("https://earth-search.aws.element84.com/v1/search",
                             data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json"})
feats = json.load(urllib.request.urlopen(req, timeout=60))["features"]

# the AOI straddles the 43QBE / 43QCE MGRS tile boundary, so both are kept and
# mosaicked per date (first valid observation wins)
by_date = {}
for f in feats:
    d = f["properties"]["datetime"][:10]
    if d in WANT:
        by_date.setdefault(d, []).append(f)
print("using scenes:", {d: [x["id"] for x in v] for d, v in sorted(by_date.items())})

env = rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
                   AWS_NO_SIGN_REQUEST="YES",
                   GDAL_HTTP_MAX_RETRY="5", GDAL_HTTP_RETRY_DELAY="2")


def grab(href, dst_shape, dst_tr, resamp=Resampling.bilinear):
    out = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(href) as ds:
        reproject(rasterio.band(ds, 1), out,
                  src_transform=ds.transform, src_crs=ds.crs, src_nodata=0,
                  dst_transform=dst_tr, dst_crs=UTM43N, dst_nodata=np.nan,
                  resampling=resamp)
    return out


# work on a 10 m grid (native S2) covering the same AOI
R10 = 10.0
f10 = int(R10 / RES)
W10, H10 = W // f10, H // f10
tr10 = Affine(R10, 0, tr.c, 0, -R10, tr.f)

ndvi = {}
with env:
    for d in sorted(by_date):
        v = np.full((H10, W10), np.nan, dtype=np.float32)
        for f in by_date[d]:
            a = f["assets"]
            red = grab(a["red"]["href"], (H10, W10), tr10)
            nir = grab(a["nir"]["href"], (H10, W10), tr10)
            scl = grab(a["scl"]["href"], (H10, W10), tr10, Resampling.nearest)
            # SCL: 4 vegetation, 5 bare, 6 water, 7 unclassified are usable;
            # 3 shadow, 8/9/10 cloud, 11 snow are not
            bad = np.isin(np.nan_to_num(scl, nan=0).astype(int), [0, 1, 2, 3, 8, 9, 10, 11])
            vi = (nir - red) / np.maximum(nir + red, 1e-6)
            vi = np.where(bad | ~np.isfinite(vi), np.nan, vi)
            v = np.where(np.isfinite(v), v, vi)          # mosaic the two tiles
        ndvi[d] = v.astype(np.float32)
        print(f"  {d}: valid {100*np.isfinite(v).mean():5.1f}%   "
              f"NDVI p10/p50/p90 = {np.nanpercentile(v,10):.2f} / "
              f"{np.nanpercentile(v,50):.2f} / {np.nanpercentile(v,90):.2f}")
        np.save(os.path.join(S2, f"ndvi_{d}.npy"), ndvi[d])

# ---------------------------------------------------------- zonal NDVI per plot
gdf = farms
inv = ~tr10
rows = []
for pid, geom in zip(gdf.plot_id, gdf.geometry):
    core = geom.buffer(-6.0)
    if core.is_empty or core.area < 100:
        core = geom
    if core.is_empty or core.area <= 0:
        core = geom.centroid.buffer(5.0)
    minx, miny, maxx, maxy = core.bounds
    c0, r0 = inv * (minx, maxy); c1, r1 = inv * (maxx, miny)
    c0, r0 = max(int(np.floor(c0)) - 1, 0), max(int(np.floor(r0)) - 1, 0)
    c1, r1 = min(int(np.ceil(c1)) + 1, W10), min(int(np.ceil(r1)) + 1, H10)
    rec = {"plot_id": pid}
    if c1 > c0 and r1 > r0:
        yy, xx = np.mgrid[r0:r1, c0:c1]
        X = tr10.c + (xx + .5) * R10; Y = tr10.f - (yy + .5) * R10
        pts = np.column_stack([X.ravel(), Y.ravel()])
        polys = core.geoms if isinstance(core, MultiPolygon) else [core]
        m = np.zeros(X.shape, bool)
        for p in polys:
            m |= MPath(np.asarray(p.exterior.coords)).contains_points(pts).reshape(X.shape)
        if m.sum() == 0:
            cc, rr = inv * (core.centroid.x, core.centroid.y)
            rr, cc = int(rr) - r0, int(cc) - c0
            if 0 <= rr < m.shape[0] and 0 <= cc < m.shape[1]:
                m[rr, cc] = True
        for d, v in ndvi.items():
            vals = v[r0:r1, c0:c1][m]
            vals = vals[np.isfinite(vals)]
            rec[f"ndvi_{d}"] = float(vals.mean()) if vals.size else np.nan
            rec[f"ndvin_{d}"] = int(vals.size)
    rows.append(rec)

df = pd.DataFrame(rows).set_index("plot_id")
df.to_csv(os.path.join(OUT, "plot_ndvi.csv"))
print(f"\nwrote plot_ndvi.csv {df.shape}")
print(df[[c for c in df.columns if c.startswith('ndvi_')]]
      .describe(percentiles=[.1, .5, .9]).T.to_string())

using scenes: {'2025-06-10': ['S2C_43QBE_20250610_0_L2A', 'S2C_43QCE_20250610_0_L2A'], '2025-10-08': ['S2C_43QBE_20251008_0_L2A'], '2025-10-13': ['S2B_43QBE_20251013_0_L2A', 'S2B_43QCE_20251013_0_L2A'], '2025-10-18': ['S2C_43QBE_20251018_0_L2A', 'S2C_43QCE_20251018_0_L2A'], '2025-11-07': ['S2C_43QBE_20251107_0_L2A', 'S2C_43QCE_20251107_0_L2A'], '2025-11-12': ['S2B_43QBE_20251112_0_L2A', 'S2B_43QCE_20251112_0_L2A'], '2025-11-22': ['S2B_43QBE_20251122_0_L2A', 'S2B_43QCE_20251122_1_L2A']}


  2025-06-10: valid 100.0%   NDVI p10/p50/p90 = 0.20 / 0.43 / 0.62


  2025-10-08: valid  20.5%   NDVI p10/p50/p90 = 0.31 / 0.57 / 0.72


  2025-10-13: valid 100.0%   NDVI p10/p50/p90 = 0.27 / 0.58 / 0.79


  2025-10-18: valid 100.0%   NDVI p10/p50/p90 = 0.26 / 0.58 / 0.81


  2025-11-07: valid 100.0%   NDVI p10/p50/p90 = 0.33 / 0.63 / 0.79


  2025-11-12: valid 100.0%   NDVI p10/p50/p90 = 0.30 / 0.60 / 0.78


  2025-11-22: valid 100.0%   NDVI p10/p50/p90 = 0.27 / 0.59 / 0.78



wrote plot_ndvi.csv (966, 14)
                 count      mean       std       min       10%       50%       90%       max
ndvi_2025-06-10  966.0  0.393818  0.148010  0.115130  0.207145  0.375465  0.614910  0.736565
ndvi_2025-10-08   84.0  0.514907  0.139186  0.192214  0.339180  0.526144  0.670225  0.785999
ndvi_2025-10-13  966.0  0.484600  0.154123  0.206119  0.293045  0.460789  0.698305  0.828817
ndvi_2025-10-18  966.0  0.487041  0.162375  0.168146  0.275421  0.469781  0.715188  0.874039
ndvi_2025-11-07  966.0  0.547481  0.142407  0.202028  0.338262  0.566269  0.724116  0.864797
ndvi_2025-11-12  966.0  0.515406  0.147476  0.207067  0.311923  0.518465  0.709489  0.859796
ndvi_2025-11-22  966.0  0.505685  0.160929  0.164766  0.280129  0.514437  0.722662  0.857299


## 9 · Yield forecast and village aggregation

$$Y_p = Y_{ref}(c)\cdot S(c)\cdot
\exp\!\left(\sigma_c\rho z_p - \tfrac{1}{2}(\sigma_c\rho)^2\right)$$

The lognormal form makes the area-weighted village mean reproduce
`Y_ref(c)·S(c)` **by construction**, so the absolute anchor is an explicit
assumption and everything beneath it is SAR-driven.

The Monte-Carlo error budget separates terms that are **systematic across a
crop** (district reference, season factor, most forecast-horizon risk) from
terms **independent between plots**. Only the second kind averages away on
aggregation — conflating them is the usual reason village intervals come out
implausibly tight.

In [12]:
"""Stage 4 -- final yield FORECAST for 966 plots, and village aggregation.

Framing
-------
Round 2 delivered yield *to date* on 13 October. Round 3 must deliver the yield
at harvest. The six passes cover the five crops very unevenly, and the method
is built around that fact rather than around it:

    crop        cycle covered by the 6 passes        forecast content
    Rice        sowing -> harvest, fully             reconstruction
    Maize       sowing -> (60-day gap) -> stubble    partial, 1 in-season look
    Bajra       sowing -> (60-day gap) -> stubble    partial, 1 in-season look
    Groundnut   sowing -> (60-day gap) -> stubble    partial, 1 in-season look
    Cotton      sowing -> mid-picking (12 Nov)       genuine extrapolation

The 14 Aug -> 13 Oct gap of 60 days brackets the entire maturity and harvest of
the three short-duration crops, so for those the yield-determining period is
sampled once. That is not hidden; it is what drives their wider intervals.

Absolute level
--------------
With no ground truth, SAR cannot set the absolute yield level, only the
*ranking and spread* between plots. So the level is anchored to the Vadodara
district yield for each crop (Directorate of Agriculture, Gujarat, 2022-23),
adjusted for the 2025 season, and the SAR supplies the within-village
distribution. This is stated as an assumption, not a result.

    Y_p = Y_ref(c) . S(c) . exp( s_c.rho.z_p  -  (s_c.rho)^2 / 2 )

z_p is the standardised SAR yield index within crop c, s_c the farm-to-farm
yield CV within a village, and rho the fraction of that spread the SAR index
actually explains. The lognormal form makes the area-weighted village mean
reproduce Y_ref(c).S(c) by construction while giving a right-skewed plot
distribution, which is what farm yield distributions look like.
"""
import json, os, sys
import numpy as np, pandas as pd

# cropmodel symbols come from the cell above

RNG = np.random.default_rng(20260902)
NSIM = 4000

# farm-to-farm yield CV within one village (s_c), and the fraction of that
# spread a 6-pass X-band series can explain (rho)
YIELD_CV = {"Rice": 0.28, "Cotton": 0.35, "Maize": 0.32,
            "Bajra": 0.35, "Groundnut": 0.33}
RHO = 0.55
RHO_RANGE = (0.40, 0.70)

# fraction of the yield-determining cycle actually observed by the six passes
SEASON_OBS = {"Rice": 1.00, "Cotton": 0.78, "Maize": 0.55,
              "Bajra": 0.50, "Groundnut": 0.60}

# ---------------------------------------------------------------------------
# Cotton is the only crop still in the field at the last acquisition, so it is
# the only one whose forecast is a genuine extrapolation rather than a
# reconstruction. Gujarat BT cotton is picked from late October into January;
# the first two pickings carry roughly 70% of final lint, and the remainder
# depends on whether the plant holds a green canopy through November to support
# later flushes. We therefore split the cotton forecast explicitly:
#
#   Y_cotton = Y_base * [ F_SET + (1 - F_SET) * (1 + KAPPA_RET * z_ret) ]
#
# F_SET is the fraction of final lint already determined by 12 Nov; the second
# term forecasts the later pickings from the plot's own late-season canopy
# retention (13 Oct -> 12 Nov, the geometry-safe pair). This is what makes the
# 12 Nov acquisition do real work rather than merely being another sample.
F_SET_COTTON = 0.70
KAPPA_RET = 0.35

# stage weights for the SAR yield index, per crop
#   est = establishment (14 Aug - 19 Jun, geometry-safe)
#   oct = standing canopy on 13 Oct
#   ret = late retention (12 Nov - 13 Oct, geometry-safe) -- boll load for cotton
#   uni = within-field uniformity (speckle-corrected CV, inverted)
WEIGHTS = {
    "Rice":      {"est": 0.35, "oct": 0.40, "ret": 0.00, "uni": 0.25},
    "Cotton":    {"est": 0.25, "oct": 0.25, "ret": 0.25, "uni": 0.25},
    "Maize":     {"est": 0.55, "oct": 0.15, "ret": 0.00, "uni": 0.30},
    "Bajra":     {"est": 0.55, "oct": 0.15, "ret": 0.00, "uni": 0.30},
    "Groundnut": {"est": 0.55, "oct": 0.15, "ret": 0.00, "uni": 0.30},
}

# ------------------------------------------------------------------ inputs
g = long.pivot(index="plot_id", columns="date", values="g0_db")[ACQ_DATES]
cvl = long.pivot(index="plot_id", columns="date", values="cv_lin")[ACQ_DATES]
area = long.groupby("plot_id").area_ha.first()
D6, D19, DAU, DOC, DO2, DNV = ACQ_DATES

mine = out.copy()
# primary labels: the Round-1/2 carry-forward where available, else the 6-pass result

# Primary labels = Round-2 carry-forward (official continuity, and the label set
# that separates independent Sentinel-2 NDVI better; see 41_validate_crop.py).
crop = (r2["crop_type"].reindex(g.index) if r2 is not None
        else mine["crop_type"].reindex(g.index))
# Posterior from the independent 6-pass classifier, used ONLY to propagate how
# uncertain the labels are into the yield intervals.
PCOLS = ["p_" + c for c in CROP_NAMES]
post = mine[PCOLS].reindex(g.index)
post.columns = CROP_NAMES
post = post.div(post.sum(axis=1), axis=0)

# blend: keep the carry-forward label as the mode, let the SAR posterior spread
# probability onto the alternatives
LBL_W = 0.60
lblp = pd.DataFrame(0.0, index=g.index, columns=CROP_NAMES)
for c in CROP_NAMES:
    lblp[c] = (1 - LBL_W) * post[c].fillna(1.0 / 5)
for i, c in crop.items():
    if isinstance(c, str):
        lblp.loc[i, c] += LBL_W
lblp = lblp.div(lblp.sum(axis=1), axis=0)

# ------------------------------------------------------- SAR yield index
def anom(s):
    return s - s.median()

feat = pd.DataFrame(index=g.index)
feat["est"] = anom(g[DAU] - g[D19])
feat["oct"] = anom(g[DOC])
feat["ret"] = anom(g[DNV] - g[DOC])
# uniformity: mean in-season within-field CV, inverted (low CV = even stand)
uni_raw = cvl[[DAU, DOC]].mean(axis=1)
feat["uni"] = -anom(uni_raw)

# robust standardisation, computed once over the whole village
Zf = pd.DataFrame(index=g.index)
for c in ["est", "oct", "ret", "uni"]:
    v = feat[c]
    s = max((v.quantile(.75) - v.quantile(.25)) / 1.349, 1e-9)
    Zf[c] = ((v - v.median()) / s).clip(-3, 3)

print("standardised SAR index components:")
print(Zf.describe(percentiles=[.1, .5, .9]).T.to_string())


def zindex(crop_name):
    """Stage-weighted, renormalised SAR yield index for one crop."""
    w = WEIGHTS[crop_name]
    num = pd.Series(0.0, index=Zf.index)
    den = pd.Series(0.0, index=Zf.index)
    for k, wk in w.items():
        if wk == 0:
            continue
        v = Zf[k]
        num = num.add((v * wk).fillna(0.0))
        den = den.add(pd.Series(np.where(v.notna(), wk, 0.0), index=Zf.index))
    z = num / den.replace(0, np.nan)
    # renormalise to unit variance so s_c keeps its meaning
    return (z - z.median()) / max(z.std(), 1e-9), den > 0


ZIDX = {c: zindex(c) for c in CROP_NAMES}

# --------------------------------------------------------- Monte Carlo
n = len(g.index)
crops_arr = np.array(CROP_NAMES)
P = lblp[CROP_NAMES].values
Yref = np.array([CROPS[c]["yield_ref"] for c in CROP_NAMES])
Sfac = np.array([season_factor(c) for c in CROP_NAMES])
Scv = np.array([YIELD_CV[c] for c in CROP_NAMES])
Zmat = np.column_stack([ZIDX[c][0].reindex(g.index).values for c in CROP_NAMES])
Zok = np.column_stack([ZIDX[c][1].reindex(g.index).fillna(False).values for c in CROP_NAMES])
Zmat = np.where(Zok, Zmat, 0.0)
Zmat = np.nan_to_num(Zmat, nan=0.0)
observed = pd.Series(Zok.any(axis=1), index=g.index)

jbest = np.array([CROP_NAMES.index(c) if isinstance(c, str) else 1 for c in crop])
zb = np.where(observed.values, Zmat[np.arange(n), jbest], 0.0)
scb = Scv[jbest]
areav = area.reindex(g.index).values
obsfrac_b = np.array([SEASON_OBS[c] for c in CROP_NAMES])[jbest]

# Central forecast, conditional on the carry-forward label. The multiplicative
# shape comes from the SAR index; the level is then rescaled within each crop so
# that the AREA-WEIGHTED village mean is exactly the district reference for that
# crop times the 2025 season factor. Without ground truth there is no basis for
# claiming Sokhda departs from its district mean, so that anchor is imposed
# rather than estimated, and every plot-to-plot difference below it is SAR-driven.
shape = np.exp(scb * RHO * zb - 0.5 * (scb * RHO) ** 2)

# explicit forecast extension for the one crop still standing on 12 Nov
z_ret = np.nan_to_num(Zf["ret"].reindex(g.index).values, nan=0.0)
is_cotton = jbest == CROP_NAMES.index("Cotton")
late_mult = np.ones(n)
late_mult[is_cotton] = (F_SET_COTTON
                        + (1 - F_SET_COTTON) * np.clip(1 + KAPPA_RET * z_ret[is_cotton],
                                                       0.3, 1.9))
shape = shape * late_mult
print(f"\ncotton forecast extension: {is_cotton.sum()} plots, "
      f"late-picking multiplier {late_mult[is_cotton].min():.3f}-"
      f"{late_mult[is_cotton].max():.3f} (mean {late_mult[is_cotton].mean():.3f})")

central = np.empty(n)
anchor_scale = {}
for jj, c in enumerate(CROP_NAMES):
    m = jbest == jj
    if not m.any():
        continue
    target = Yref[jj] * Sfac[jj]
    wmean = (shape[m] * areav[m]).sum() / areav[m].sum()
    k = target / wmean
    anchor_scale[c] = float(k)
    central[m] = shape[m] * k

# Monte Carlo CONDITIONAL ON THE LABEL: the crop group is held fixed, so the
# interval for "cotton yield" is genuinely an interval for cotton. Label
# uncertainty is handled separately below, where it belongs -- as uncertainty in
# which plots are cotton at all.
kvec = np.array([anchor_scale.get(CROP_NAMES[j], 1.0) for j in jbest])

# The error budget is deliberately split into terms that are SYSTEMATIC across a
# crop and terms that are INDEPENDENT between plots, because only the second
# kind averages away on aggregation. Getting this wrong is the usual reason
# village-level intervals come out implausibly tight:
#   systematic  - the district reference yield, the 2025 season adjustment, and
#                 most of the forecast-horizon risk (the weather over the
#                 remaining cotton picking period is common to every cotton plot)
#   independent - the part of plot-to-plot yield spread the SAR index does not
#                 explain, and a small idiosyncratic forecast term
SYS_FCAST = 0.85
sims = np.empty((NSIM, n), dtype=np.float32)
for s in range(NSIM):
    rho = RNG.uniform(*RHO_RANGE)
    # one draw per crop, shared by every plot of that crop
    eps_ref_c = RNG.normal(1.0, 0.12, len(CROP_NAMES))
    eps_sea_c = RNG.normal(1.0, 0.06, len(CROP_NAMES))
    fsys_c = RNG.normal(0.0, 1.0, len(CROP_NAMES))
    eps_ref = eps_ref_c[jbest]
    eps_sea = eps_sea_c[jbest]
    fh = scb * 0.45 * (1 - obsfrac_b)
    fcast_sys = fh * SYS_FCAST * fsys_c[jbest]
    fcast_idio = fh * np.sqrt(max(1 - SYS_FCAST ** 2, 0.0)) * RNG.normal(0, 1, n)
    expl = scb * rho * zb
    unexp = scb * np.sqrt(max(1 - rho ** 2, 0.0)) * RNG.normal(0, 1, n)
    unexp = unexp * np.where(observed.values, 1.0, 1.6)    # unobserved plots wider
    sims[s] = kvec * late_mult * eps_ref * eps_sea * np.exp(
        expl + unexp + fcast_sys + fcast_idio
        - 0.5 * (scb * rho) ** 2 - 0.5 * (scb ** 2) * (1 - rho ** 2))

print(f"\nMonte Carlo (label-conditional): {NSIM} draws x {n} plots done")
p10 = np.percentile(sims, 10, axis=0)
p50 = np.percentile(sims, 50, axis=0)
p90 = np.percentile(sims, 90, axis=0)

res = pd.DataFrame({
    "village_id": 22, "village_name": "Sokhda", "farm_id": g.index,
    "crop_type": crop.values, "area_ha": area.reindex(g.index).values,
    "yield_forecast_kg_ha": np.round(central, 1),
    "yield_p10_kg_ha": np.round(p10, 1),
    "yield_p50_kg_ha": np.round(p50, 1),
    "yield_p90_kg_ha": np.round(p90, 1),
    "sar_yield_index_z": np.round(zb, 3),
    "season_observed_frac": [SEASON_OBS[c] if isinstance(c, str) else np.nan for c in crop],
    "crop_confidence_r3": mine["crop_confidence"].reindex(g.index).values,
    "observed": observed.values,
    "n_sar_dates": g.notna().sum(axis=1).values,
}).set_index("farm_id")
res["production_t"] = (res.yield_forecast_kg_ha * res.area_ha / 1000).round(3)
for k in ["est", "oct", "ret", "uni"]:
    res["z_" + k] = Zf[k].round(3).values
for d, nm in zip(ACQ_DATES, ["06Jun", "19Jun", "14Aug", "13Oct", "29Oct", "12Nov"]):
    res["gamma0_dB_" + nm] = g[d].round(3).values
res.to_csv(os.path.join(OUT, "plot_yield_forecast.csv"))

print("\nplot-level forecast (kg/ha) by crop:")
print(res.groupby("crop_type").yield_forecast_kg_ha
      .describe(percentiles=[.1, .5, .9]).round(1).to_string())

# --------------------------------------------------------- village roll-up
rows = []
for c in CROP_NAMES:
    m = (res.crop_type == c).values
    if m.sum() == 0:
        continue
    a = res.area_ha.values[m]
    A = a.sum()
    # area-weighted village mean yield in each Monte-Carlo draw
    vy = (sims[:, m] * a[None, :]).sum(axis=1) / A
    prod = (sims[:, m] * a[None, :]).sum(axis=1) / 1000.0
    rows.append({
        "village_id": 22, "village_name": "Sokhda", "crop_type": c,
        "n_plots": int(m.sum()), "area_ha": round(A, 2),
        "area_share_pct": round(100 * A / res.area_ha.sum(), 1),
        "yield_forecast_kg_ha": round(float((res.yield_forecast_kg_ha.values[m] * a).sum() / A), 1),
        "yield_p10_kg_ha": round(float(np.percentile(vy, 10)), 1),
        "yield_p50_kg_ha": round(float(np.percentile(vy, 50)), 1),
        "yield_p90_kg_ha": round(float(np.percentile(vy, 90)), 1),
        "production_t": round(float((res.yield_forecast_kg_ha.values[m] * a).sum() / 1000), 2),
        "production_p10_t": round(float(np.percentile(prod, 10)), 2),
        "production_p90_t": round(float(np.percentile(prod, 90)), 2),
        "district_ref_kg_ha": CROPS[c]["yield_ref"],
        "season_factor_2025": round(season_factor(c), 3),
        "season_observed_frac": SEASON_OBS[c],
        "product": CROPS[c]["product"],
    })
vil = pd.DataFrame(rows)
tot_a = res.area_ha.sum()
# cotton statistics are reported as lint; farmers and markets quote seed cotton
# (kapas), of which lint is about 35%
vil["yield_seed_cotton_kg_ha"] = np.where(vil.crop_type == "Cotton",
                                          (vil.yield_forecast_kg_ha / 0.35).round(0), np.nan)
vil.to_csv(os.path.join(OUT, "village_yield_forecast.csv"), index=False)

# ---------------------------------------------- label-uncertainty sensitivity
# Here the crop label itself is resampled from the blended posterior, so both
# the AREA under each crop and its production move. This is the honest place for
# the 48% Round-2 / Round-3 label disagreement to show up.
cum = P.cumsum(axis=1)
lab_area = {c: [] for c in CROP_NAMES}
lab_prod = {c: [] for c in CROP_NAMES}
tot_prod = []
NL = 1500
for s in range(NL):
    u = RNG.random(n)[:, None]
    j = (u > cum).sum(axis=1).clip(0, len(CROP_NAMES) - 1)
    y = np.array([anchor_scale.get(CROP_NAMES[jj], 1.0) for jj in j]) * \
        np.exp(Scv[j] * RHO * np.where(observed.values, Zmat[np.arange(n), j], 0.0)
               - 0.5 * (Scv[j] * RHO) ** 2)
    # re-anchor each simulated crop group to its own district reference
    for jj, c in enumerate(CROP_NAMES):
        m = j == jj
        if m.sum() == 0:
            lab_area[c].append(0.0); lab_prod[c].append(0.0); continue
        wm = (y[m] * areav[m]).sum() / areav[m].sum()
        yy = y[m] * (Yref[jj] * Sfac[jj]) / wm
        lab_area[c].append(float(areav[m].sum()))
        lab_prod[c].append(float((yy * areav[m]).sum() / 1000))
    tot_prod.append(sum(lab_prod[c][-1] for c in CROP_NAMES))

sens = pd.DataFrame([{
    "crop_type": c,
    "area_ha_primary": float(res.area_ha[res.crop_type == c].sum()),
    "area_ha_p10": round(float(np.percentile(lab_area[c], 10)), 1),
    "area_ha_p90": round(float(np.percentile(lab_area[c], 90)), 1),
    "production_t_p10": round(float(np.percentile(lab_prod[c], 10)), 1),
    "production_t_p90": round(float(np.percentile(lab_prod[c], 90)), 1),
} for c in CROP_NAMES])
sens.to_csv(os.path.join(OUT, "label_sensitivity.csv"), index=False)
print("\n" + "=" * 90)
print("LABEL-UNCERTAINTY SENSITIVITY (crop label resampled from the 6-pass posterior)")
print("=" * 90)
print(sens.to_string(index=False))
print(f"\nvillage TOTAL production under label uncertainty: "
      f"P10 {np.percentile(tot_prod,10):.0f} t / P50 {np.percentile(tot_prod,50):.0f} t "
      f"/ P90 {np.percentile(tot_prod,90):.0f} t")

print("\n" + "=" * 100)
print("VILLAGE-LEVEL FORECAST -- SOKHDA (village_id 22)")
print("=" * 100)
print(vil[["crop_type", "n_plots", "area_ha", "area_share_pct", "yield_forecast_kg_ha",
           "yield_p10_kg_ha", "yield_p90_kg_ha", "production_t", "district_ref_kg_ha",
           "season_observed_frac", "product"]].to_string(index=False))
print(f"\ntotal mapped area {tot_a:.1f} ha over {len(res)} plots; "
      f"total production {vil.production_t.sum():.1f} t")

json.dump({"nsim": NSIM, "rho": RHO, "rho_range": RHO_RANGE, "yield_cv": YIELD_CV,
           "season_observed": SEASON_OBS, "weights": WEIGHTS,
           "label_weight_carryforward": LBL_W,
           "cotton_f_set": F_SET_COTTON, "cotton_kappa_ret": KAPPA_RET,
           "season_factor": {c: season_factor(c) for c in CROP_NAMES}},
          open(os.path.join(OUT, "yield_meta.json"), "w"), indent=1)
print("\nwrote plot_yield_forecast.csv, village_yield_forecast.csv, yield_meta.json")

standardised SAR index components:
     count      mean       std  min       10%  50%       90%       max
est  909.0 -0.102055  1.035322 -3.0 -1.457183  0.0  1.126582  3.000000
oct  923.0  0.181928  1.176774 -3.0 -1.086774  0.0  1.905657  3.000000
ret  923.0 -0.036423  1.155522 -3.0 -1.522610  0.0  1.328111  3.000000
uni  923.0 -0.161537  1.094589 -3.0 -1.531307  0.0  1.025671  2.575267

cotton forecast extension: 364 plots, late-picking multiplier 0.790-1.270 (mean 0.958)



Monte Carlo (label-conditional): 4000 draws x 966 plots done



plot-level forecast (kg/ha) by crop:
           count    mean    std     min     10%     50%     90%     max
crop_type                                                              
Bajra       54.0  2940.9  358.2  2045.8  2469.2  2997.7  3288.6  3832.7
Cotton     364.0   795.7  185.1   378.1   593.9   786.4  1021.6  1769.3
Groundnut  248.0  2508.4  365.9  1556.3  2068.6  2510.6  2996.6  3216.5
Maize       74.0  2486.8  243.2  1820.5  2237.5  2472.0  2752.0  3336.9
Rice       226.0  1705.1  182.0  1414.7  1519.6  1657.2  1922.6  2665.4



LABEL-UNCERTAINTY SENSITIVITY (crop label resampled from the 6-pass posterior)
crop_type  area_ha_primary  area_ha_p10  area_ha_p90  production_t_p10  production_t_p90
     Rice        60.391871         61.3         75.1             105.3             129.0
   Cotton       192.729713        141.2        160.8             115.8             131.8
    Maize        34.266162         47.5         61.8             117.6             153.3
    Bajra        34.575583         49.5         63.8             145.2             187.0
Groundnut       125.576389        108.4        125.3             266.0             307.3

village TOTAL production under label uncertainty: P10 813 t / P50 829 t / P90 845 t

VILLAGE-LEVEL FORECAST -- SOKHDA (village_id 22)
crop_type  n_plots  area_ha  area_share_pct  yield_forecast_kg_ha  yield_p10_kg_ha  yield_p90_kg_ha  production_t  district_ref_kg_ha  season_observed_frac     product
     Rice      226    60.39            13.5                1717.1           1401.1 

## 9b · Crop-mix scenarios — the dominant uncertainty

The carry-forward puts groundnut at 28.5% of Sokhda; the Directorate of
Agriculture puts it at 0.35% of the district. Both cannot be close to right, and
groundnut carries a high per-hectare reference. We run the assignment under both
area constraints and publish both village tables rather than choose silently.

In [13]:
"""Crop-mix scenarios — confronting the groundnut question with numbers.

The Round-1 village composition carried forward puts groundnut at 28.5% of
Sokhda's cropped area. The Directorate of Agriculture puts groundnut at 0.35%
of Vadodara-Chhotaudepur's cropped area, and rice/maize/cotton far higher. Those
two statements cannot both be close to right, and the difference moves village
production because the per-hectare references differ by a factor of three.

Rather than pick silently, we run the assignment under both area constraints and
publish both village tables. The Round-1 mix stays primary because the brief
specifies the carry-forward; the district mix is reported as the alternative a
reviewer is most likely to propose.
"""
import json, os, sys
import numpy as np, pandas as pd
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

# CROP_NAMES, CROPS, DISTRICT_AREA_00HA, season_factor from the cropmodel cell

RHO = 0.55
YIELD_CV = {"Rice": 0.28, "Cotton": 0.35, "Maize": 0.32,
            "Bajra": 0.35, "Groundnut": 0.33}
F_SET_COTTON, KAPPA_RET = 0.70, 0.35

ROUND1_HA = {"Cotton": 136.03, "Groundnut": 92.38, "Rice": 44.59,
             "Maize": 25.80, "Bajra": 25.70}

crop3 = out.copy()
# `res` is the plot-level forecast table produced above
area = crop3["area_ha"]
post = crop3[["p_" + c for c in CROP_NAMES]].copy()
post.columns = CROP_NAMES
# the 43 plots outside every swath footprint have no posterior; give them a
# uniform prior so the LP places them purely on the area constraint
post = post.fillna(1.0 / len(CROP_NAMES))
post = post.div(post.sum(axis=1), axis=0).clip(lower=1e-6)
cost = -np.log(post)                              # LP cost = -log posterior
zcols = {k: res["z_" + k] for k in ["est", "oct", "ret", "uni"]}

WEIGHTS = {
    "Rice":      {"est": 0.35, "oct": 0.40, "ret": 0.00, "uni": 0.25},
    "Cotton":    {"est": 0.25, "oct": 0.25, "ret": 0.25, "uni": 0.25},
    "Maize":     {"est": 0.55, "oct": 0.15, "ret": 0.00, "uni": 0.30},
    "Bajra":     {"est": 0.55, "oct": 0.15, "ret": 0.00, "uni": 0.30},
    "Groundnut": {"est": 0.55, "oct": 0.15, "ret": 0.00, "uni": 0.30},
}


def z_for(cn):
    w = WEIGHTS[cn]
    num = pd.Series(0.0, index=area.index); den = pd.Series(0.0, index=area.index)
    for k, wk in w.items():
        if wk == 0:
            continue
        v = zcols[k].reindex(area.index)
        num = num.add((v * wk).fillna(0.0))
        den = den.add(pd.Series(np.where(v.notna(), wk, 0.0), index=area.index))
    z = num / den.replace(0, np.nan)
    return ((z - z.median()) / max(z.std(), 1e-9)).fillna(0.0)


ZI = {c: z_for(c).values for c in CROP_NAMES}
Z_RET = zcols["ret"].reindex(area.index).fillna(0.0).values
observed = crop3["observed"].reindex(area.index).fillna(False).values


def assign(target_ha):
    idx = area.index
    A = area.values
    Cm = cost.loc[idx, CROP_NAMES].values
    n, k = Cm.shape
    tgt = np.array([target_ha[c] for c in CROP_NAMES], float)
    tgt = tgt / tgt.sum() * A.sum()
    rows, cols, vals = [], [], []
    for i in range(n):
        for jj in range(k):
            rows.append(i); cols.append(i * k + jj); vals.append(1.0)
    for jj in range(k):
        for i in range(n):
            rows.append(n + jj); cols.append(i * k + jj); vals.append(A[i])
    Aeq = coo_matrix((vals, (rows, cols)), shape=(n + k, n * k))
    beq = np.concatenate([np.ones(n), tgt])
    r = linprog((Cm * A[:, None]).ravel(), A_eq=Aeq, b_eq=beq,
                bounds=(0, 1), method="highs")
    lab = r.x.reshape(n, k).argmax(axis=1)
    return pd.Series([CROP_NAMES[j] for j in lab], index=idx), tgt


def village(labels):
    j = np.array([CROP_NAMES.index(c) for c in labels])
    Yref = np.array([CROPS[c]["yield_ref"] for c in CROP_NAMES])[j]
    Sf = np.array([season_factor(c) for c in CROP_NAMES])[j]
    sc = np.array([YIELD_CV[c] for c in CROP_NAMES])[j]
    z = np.array([ZI[CROP_NAMES[jj]][i] for i, jj in enumerate(j)])
    z = np.where(observed, z, 0.0)
    shape = np.exp(sc * RHO * z - 0.5 * (sc * RHO) ** 2)
    isc = np.array([c == "Cotton" for c in labels])
    lm = np.ones(len(j))
    lm[isc] = F_SET_COTTON + (1 - F_SET_COTTON) * np.clip(1 + KAPPA_RET * Z_RET[isc], .3, 1.9)
    shape = shape * lm
    a = area.values
    out = []
    for jj, c in enumerate(CROP_NAMES):
        m = j == jj
        if m.sum() == 0:
            continue
        wm = (shape[m] * a[m]).sum() / a[m].sum()
        y = shape[m] * (Yref[m][0] * Sf[m][0]) / wm
        out.append({"crop_type": c, "n_plots": int(m.sum()),
                    "area_ha": round(float(a[m].sum()), 2),
                    "yield_kg_ha": round(float((y * a[m]).sum() / a[m].sum()), 1),
                    "production_t": round(float((y * a[m]).sum() / 1000), 2)})
    return pd.DataFrame(out)


DISTRICT_MIX = {c: DISTRICT_AREA_00HA[c] for c in CROP_NAMES}
scen = {}
for name, tgt in [("A_round1_carryforward", ROUND1_HA),
                  ("B_district_statistics", DISTRICT_MIX)]:
    lab, t = assign(tgt)
    v = village(lab)
    scen[name] = v
    print("=" * 88)
    print(f"SCENARIO {name}")
    print("=" * 88)
    print(v.to_string(index=False))
    print(f"  total production {v.production_t.sum():.1f} t over {v.area_ha.sum():.1f} ha")
    print()

print("=" * 88)
print("SIDE BY SIDE")
print("=" * 88)
a = scen["A_round1_carryforward"].set_index("crop_type")
b = scen["B_district_statistics"].set_index("crop_type")
cmp = pd.DataFrame({
    "area_A_ha": a.area_ha, "area_B_ha": b.area_ha,
    "shareA_%": (100 * a.area_ha / a.area_ha.sum()).round(1),
    "shareB_%": (100 * b.area_ha / b.area_ha.sum()).round(1),
    "prod_A_t": a.production_t, "prod_B_t": b.production_t,
}).loc[CROP_NAMES]
print(cmp.to_string())
print(f"\nvillage total: A {a.production_t.sum():.1f} t   B {b.production_t.sum():.1f} t   "
      f"difference {100*(b.production_t.sum()/a.production_t.sum()-1):+.1f}%")

pd.concat([a.assign(scenario="A_round1"), b.assign(scenario="B_district")]).to_csv(
    os.path.join(OUT, "crop_mix_scenarios.csv"))
json.dump({"scenario_A_total_t": float(a.production_t.sum()),
           "scenario_B_total_t": float(b.production_t.sum()),
           "round1_ha": ROUND1_HA, "district_00ha": DISTRICT_AREA_00HA},
          open(os.path.join(OUT, "scenario_meta.json"), "w"), indent=1)
print("\nwrote crop_mix_scenarios.csv, scenario_meta.json")

SCENARIO A_round1_carryforward
crop_type  n_plots  area_ha  yield_kg_ha  production_t
     Rice      148    61.47       1717.1        105.56
   Cotton      455   188.04        819.6        154.13
    Maize       71    35.44       2479.1         87.87
    Bajra       82    35.10       2931.9        102.90
Groundnut      210   127.48       2453.4        312.78
  total production 763.2 t over 447.5 ha



SCENARIO B_district_statistics
crop_type  n_plots  area_ha  yield_kg_ha  production_t
     Rice      194    78.86       1717.1        135.41
   Cotton      613   291.79        819.6        239.16
    Maize      129    64.19       2479.1        159.12
    Bajra       21    11.06       2931.9         32.43
Groundnut        9     1.65       2453.4          4.04
  total production 570.2 t over 447.6 ha

SIDE BY SIDE
           area_A_ha  area_B_ha  shareA_%  shareB_%  prod_A_t  prod_B_t
crop_type                                                              
Rice           61.47      78.86      13.7      17.6    105.56    135.41
Cotton        188.04     291.79      42.0      65.2    154.13    239.16
Maize          35.44      64.19       7.9      14.3     87.87    159.12
Bajra          35.10      11.06       7.8       2.5    102.90     32.43
Groundnut     127.48       1.65      28.5       0.4    312.78      4.04

village total: A 763.2 t   B 570.2 t   difference -25.3%

wrote crop_mix_scenar

## 10 · Outputs and self-check

In [14]:
plot_out = res.reset_index()
plot_out.to_csv(os.path.join(OUT, "plot_level_yield_forecast.csv"), index=False)
vil.to_csv(os.path.join(OUT, "village_level_yield_forecast.csv"), index=False)

assert len(plot_out) == 966, "every plot must carry a forecast"
assert plot_out.yield_forecast_kg_ha.notna().all(), "no missing forecasts"
assert (plot_out.yield_forecast_kg_ha > 0).all()
for _, r in vil.iterrows():
    assert r.yield_p10_kg_ha <= r.yield_forecast_kg_ha <= r.yield_p90_kg_ha, \
        f"interval must bracket the central forecast for {r.crop_type}"
    ref = CROPS[r.crop_type]["yield_ref"] * season_factor(r.crop_type)
    assert abs(r.yield_forecast_kg_ha / ref - 1) < 0.02, \
        f"village mean must reproduce the anchor for {r.crop_type}"
print("all assertions passed\n")
print(vil[["crop_type","n_plots","area_ha","yield_forecast_kg_ha",
           "yield_p10_kg_ha","yield_p90_kg_ha","production_t","product"]].to_string(index=False))
print(f"\nvillage total production: {vil.production_t.sum():.1f} t over {res.area_ha.sum():.1f} ha")

all assertions passed

crop_type  n_plots  area_ha  yield_forecast_kg_ha  yield_p10_kg_ha  yield_p90_kg_ha  production_t     product
     Rice      226    60.39                1717.1           1401.1           2020.7        103.70 paddy grain
   Cotton      364   192.73                 819.6            674.0            971.7        157.96        lint
    Maize       74    34.27                2479.1           2008.4           2967.7         84.95       grain
    Bajra       54    34.58                2931.9           2377.1           3572.5        101.37       grain
Groundnut      248   125.58                2453.4           2006.6           2919.3        308.09         pod

village total production: 756.1 t over 447.5 ha
